# In-House Layout Clustering Pipeline

**목적**: Layout feature 126개 × **64,955 rows** 데이터에서  
- CD (nm) 산포를 최소화하는 클러스터 도출  
- SHAP 기반 feature 중요도 분석 및 선택  
- **Cost function**: `α × mean(4σ range %) + β × max(4σ range %)` 최소화  

**4σ range % 정의 (percentile 기반)**:
```
1. CD_norm    = CD / median_CD               # median_CD로 정규화
2. lower_norm = percentile(CD_norm, 0.00315) # 4σ 하한 (±4σ ≈ 0.00315 / 99.99685 pct)
3. upper_norm = percentile(CD_norm, 99.99685)# 4σ 상한
4. 4σ range % = (upper_norm - lower_norm) × 100
```

- **제약**: 클러스터당 최소 샘플 수 ≥ MIN_CLUSTER_COUNT  
- **median alignment 효과 분석**: 클러스터별 CD median을 전체 median으로 보정 후 4σ range % 개선량 확인  

**알고리즘 10종 비교**:  
1. Decision Tree Clustering  
2. K-Means (MiniBatch, SHAP feature 기반)  
3. Autoencoder + K-Means (DL 잠재 표현 기반)  
4. GMM (Gaussian Mixture Model)  
5. Bisecting K-Means  
6. Agglomerative Ward Clustering  
7. HDBSCAN (밀도 기반, 자동 클러스터 수)  
8. Spectral Clustering  
9. IsolationForest + K-Means (이상치 강인)  
10. VAE + K-Means (Variational Autoencoder 잠재 표현)


## 0. 설정 (Configuration)

In [ ]:
# ============================================================
#  GLOBAL CONFIG — 이 셀만 수정하면 전체 파이프라인 제어 가능
# ============================================================

DATA_PATH  = None           # None → 합성 데이터 자동 생성
# DATA_PATH = 'data/layout_features.csv'

CD_COLUMN  = 'CD (nm)'      # CD 컬럼명
N_ROWS     = 64_955         # 실제 데이터 행 수 (합성 데이터도 동일 크기로 생성)
N_FEAT     = 126

# ── 4σ range % 계산 기준 percentile ─────────────────────────
LOWER_PCT  = 0.00315        # 4σ lower percentile
UPPER_PCT  = 99.99685       # 4σ upper percentile

# ── SHAP ────────────────────────────────────────────────────
SHAP_TOP_K       = 20
SHAP_SAMPLE_N    = 10_000
XGB_N_ESTIMATORS = 200
XGB_MAX_DEPTH    = 6

# ── 클러스터 제약 ───────────────────────────────────────────
MIN_CLUSTER_COUNT = 100     # 클러스터 최소 샘플 수

# ── Optuna 최적화 설정 ────────────────────────────────────────
OPTUNA_N_TRIALS    = 50    # 기본 trial 수 (빠른 방법)
OPTUNA_N_TRIALS_DL = 10    # AE/VAE 등 느린 방법
OPTUNA_N_JOBS      = 1     # 병렬 trial 수 (1=순차, -1=CPU 전체 활용)

# ── Cost function 모드 ──────────────────────────────────────────
# "combined"        : combined 4σ range after alignment (기본값)
# "soft_penalty"    : combined_4σ + LAMBDA_PENALTY × weighted_mean(per_cluster_4σ%)
# "hard_constraint" : max(per_cluster_4σ%) > baseline × THRESHOLD_RATIO → inf
COST_MODE                          = "combined"
LAMBDA_PENALTY                     = 0.3
MAX_CLUSTER_4SIGMA_THRESHOLD_RATIO = 0.8

# ── Grid search: k = 5 ~ 101, step 1 ─────────────────────────
K_RANGE        = list(range(5, 102, 1))   # 모든 K-기반 방법 공통
DT_MAX_LEAF_RANGE = list(range(5, 102, 1))
HDBSCAN_MIN_SIZES = list(range(MIN_CLUSTER_COUNT, MIN_CLUSTER_COUNT * 6, MIN_CLUSTER_COUNT))

# ── Autoencoder / VAE 설정 ───────────────────────────────────
AE_LATENT_DIM  = 16
AE_EPOCHS      = 40
AE_BATCH_SIZE  = 1024
AE_LR          = 1e-3
AE_PATIENCE    = 5
VAE_BETA       = 1.0

# ── 기타 ─────────────────────────────────────────────────────
RANDOM_STATE = 42
OUTPUT_DIR   = 'output'

import os, warnings
warnings.filterwarnings('ignore')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Config loaded.')
print(f'  CD column        : {CD_COLUMN}')
print(f'  Data rows        : {N_ROWS:,}')
print(f'  4σ percentile    : [{LOWER_PCT}, {UPPER_PCT}]')
print(f'  Cost mode        : {COST_MODE}')
if COST_MODE == "soft_penalty":
    print(f'  Lambda penalty   : {LAMBDA_PENALTY}')
elif COST_MODE == "hard_constraint":
    print(f'  Threshold ratio  : {MAX_CLUSTER_4SIGMA_THRESHOLD_RATIO}')
print(f'  Min cluster cnt  : {MIN_CLUSTER_COUNT}')
print(f'  Grid search K    : {K_RANGE[0]}~{K_RANGE[-1]} (step 1, {len(K_RANGE)} values)')
print(f'  Optuna trials    : {OPTUNA_N_TRIALS} (DL: {OPTUNA_N_TRIALS_DL}), n_jobs={OPTUNA_N_JOBS}')


## 1. 라이브러리 임포트

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from tqdm.auto import tqdm
from pathlib import Path
from collections import Counter
import json, joblib

from sklearn.preprocessing import RobustScaler
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

import xgboost as xgb
import shap

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except ImportError:
    from optimization.common import optuna_compat as optuna

shap.initjs()
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch device: {DEVICE}')
print('Libraries loaded.')

## 2. 데이터 로딩 & EDA

In [ ]:
def generate_synthetic_data(n_rows, n_feat, cd_col, seed):
    rng = np.random.default_rng(seed)
    group = rng.choice([0,1,2,3,4], size=n_rows, p=[0.35,0.25,0.20,0.12,0.08])
    X = rng.standard_normal((n_rows, n_feat)).astype(np.float32)
    for g in range(5):
        mask = group == g
        X[mask] = X[mask] * rng.uniform(0.5,1.5,n_feat) + rng.uniform(-2,2,n_feat)
    X[rng.random((n_rows, n_feat)) < 0.01] = np.nan
    feat_names = [f'feat_{i:03d}' for i in range(n_feat)]
    df = pd.DataFrame(X, columns=feat_names)
    coef = rng.uniform(-1, 1, n_feat)
    cd_base = df.fillna(0).values @ coef * 0.3
    group_offset = np.array([50, 70, 90, 110, 130])[group]
    noise_std    = np.array([3,  5,  4,  6,   8  ])[group]
    df[cd_col] = (cd_base + group_offset + rng.normal(0, noise_std, n_rows)).astype(np.float32)
    df['_true_group'] = group
    return df

if DATA_PATH is None:
    print(f'합성 데이터 생성 중 ({N_ROWS:,} rows × {N_FEAT} feats)...')
    df_raw = generate_synthetic_data(N_ROWS, N_FEAT, CD_COLUMN, RANDOM_STATE)
else:
    df_raw = pd.read_csv(DATA_PATH)

print(f'Shape: {df_raw.shape}')
df_raw.head(3)

In [ ]:
feat_cols = [c for c in df_raw.columns if c not in [CD_COLUMN, '_true_group']]

# ── EDA용 4σ 경계: percentile 기반 ───────────────────────────
cd_eda      = df_raw[CD_COLUMN].dropna().values
med_eda     = float(np.median(cd_eda))
cd_norm_eda = cd_eda / med_eda
lower_norm  = np.percentile(cd_norm_eda, LOWER_PCT)
upper_norm  = np.percentile(cd_norm_eda, UPPER_PCT)
lower_cd    = lower_norm * med_eda
upper_cd    = upper_norm * med_eda
range_pct   = (upper_norm - lower_norm) * 100

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# ── Plot 1: CD 분포 + 4σ 경계 ────────────────────────────────
axes[0].hist(cd_eda, bins=120, color='steelblue', edgecolor='none', alpha=0.75, density=True)
axes[0].axvline(upper_cd, color='red',    ls='--', lw=1.8, label=f'4σ upper = {upper_cd:.2f} nm')
axes[0].axvline(lower_cd, color='orange', ls='--', lw=1.8, label=f'4σ lower = {lower_cd:.2f} nm')
axes[0].axvline(med_eda,  color='navy',   ls=':',  lw=1.5, label=f'median = {med_eda:.2f} nm')
axes[0].axvspan(lower_cd, upper_cd, alpha=0.08, color='red')
axes[0].annotate(
    f'4σ range = {range_pct:.3f} %\n'
    f'pct [{LOWER_PCT}, {UPPER_PCT}]\n'
    f'on CD / {med_eda:.2f} nm',
    xy=(0.97, 0.97), xycoords='axes fraction',
    ha='right', va='top', fontsize=8.5,
    bbox=dict(boxstyle='round,pad=0.4', fc='lightyellow', ec='gray', alpha=0.92),
)
axes[0].set_title('CD (nm) Distribution with 4σ Bounds', fontweight='bold')
axes[0].set_xlabel('CD (nm)'); axes[0].set_ylabel('Density')
axes[0].legend(fontsize=8)

# ── Plot 2: CD Empirical CDF + percentile 마커 ───────────────
sorted_cd_norm = np.sort(cd_norm_eda)
cdf            = np.arange(1, len(sorted_cd_norm) + 1) / len(sorted_cd_norm) * 100
axes[1].plot(sorted_cd_norm * med_eda, cdf, 'steelblue', lw=1.5)
axes[1].axhline(LOWER_PCT,  color='orange', ls='--', lw=1.5, label=f'lower pct={LOWER_PCT}%')
axes[1].axhline(UPPER_PCT,  color='red',    ls='--', lw=1.5, label=f'upper pct={UPPER_PCT}%')
axes[1].axvline(lower_cd,   color='orange', ls=':',  lw=1.2)
axes[1].axvline(upper_cd,   color='red',    ls=':',  lw=1.2)
axes[1].axvspan(lower_cd, upper_cd, alpha=0.06, color='red')
axes[1].set_title('CD Empirical CDF with 4σ Percentile Markers', fontweight='bold')
axes[1].set_xlabel('CD (nm)'); axes[1].set_ylabel('Cumulative %')
axes[1].legend(fontsize=8)

# ── Plot 3: Feature-CD 피어슨 상관관계 Top 20 ────────────────
feat_var = df_raw[feat_cols].var()
corr_cd  = df_raw[feat_cols].corrwith(df_raw[CD_COLUMN]).abs().dropna().sort_values(ascending=False)
top_corr = corr_cd.head(20)
colors_c = plt.cm.RdYlGn(top_corr.values / top_corr.values.max())
axes[2].barh(top_corr.index[::-1], top_corr.values[::-1], color=colors_c[::-1])
axes[2].set_title('Top 20 Feature |Pearson Corr| with CD', fontweight='bold')
axes[2].set_xlabel('|Correlation with CD (nm)|')
axes[2].axvline(0.1, color='gray', ls=':', lw=1, label='0.1 threshold')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/01_eda.png', bbox_inches='tight')
plt.show()

print(df_raw[CD_COLUMN].describe())
print(f'\n4σ range (percentile 기반):')
print(f'  median CD   : {med_eda:.4f} nm')
print(f'  4σ_upper    : {upper_cd:.4f} nm  (norm={upper_norm:.6f}, pct={UPPER_PCT})')
print(f'  4σ_lower    : {lower_cd:.4f} nm  (norm={lower_norm:.6f}, pct={LOWER_PCT})')
print(f'  4σ range %  : {range_pct:.4f} %')


## 3. 전처리

In [ ]:
# 저분산 feature 제거
feat_cols_clean = feat_var[feat_var >= 1e-6].index.tolist()
print(f'저분산 제거 후 feature 수: {len(feat_cols_clean)} (제거: {len(feat_cols)-len(feat_cols_clean)}개)')

# CD 결측 제거
df_clean = df_raw.dropna(subset=[CD_COLUMN]).copy()

# Feature imputation & scaling
X_df = df_clean[feat_cols_clean].fillna(df_clean[feat_cols_clean].median())
y    = df_clean[CD_COLUMN].values.astype(np.float64)

scaler   = RobustScaler()
X_scaled = scaler.fit_transform(X_df)
X_scaled_df = pd.DataFrame(X_scaled, columns=feat_cols_clean, index=X_df.index)

# ── 전체 기준값 (percentile 기반 4σ range %) ─────────────────
OVERALL_MEDIAN_CD = float(np.median(y))
y_norm = y / OVERALL_MEDIAN_CD              # 정규화: median = 1

# percentile로 4σ upper/lower 계산
BASELINE_LOWER_NORM = float(np.percentile(y_norm, LOWER_PCT))
BASELINE_UPPER_NORM = float(np.percentile(y_norm, UPPER_PCT))
BASELINE_4SIGMA_RANGE_PCT = (BASELINE_UPPER_NORM - BASELINE_LOWER_NORM) * 100

print(f'\n전처리 완료: X={X_scaled.shape}, y={y.shape}')
print(f'  전체 CD median            : {OVERALL_MEDIAN_CD:.4f} nm')
print(f'  4σ_upper (norm, pct={UPPER_PCT}): {BASELINE_UPPER_NORM:.6f}')
print(f'  4σ_lower (norm, pct={LOWER_PCT}) : {BASELINE_LOWER_NORM:.6f}')
print(f'  기준선 4σ range %          : {BASELINE_4SIGMA_RANGE_PCT:.4f} %')
print(f'  데이터 행 수               : {len(y):,}')


## 4. Cost Function (4σ range % — Percentile 기반)

```
CD_norm    = CD / median_CD                        # Step 1: 정규화 (median = 1)
lower_norm = percentile(CD_norm, 0.00315)          # Step 2: 4σ lower bound
upper_norm = percentile(CD_norm, 99.99685)         # Step 3: 4σ upper bound
4σ range % = (upper_norm - lower_norm) × 100      # Step 4: range as %
```

> **Why percentile?** 정규분포 가정 없이 실제 데이터 분포를 그대로 반영.  
> 0.00315% / 99.99685% 는 ±4σ 정규분포 대응 분위수.

Cost = `α × weighted_mean(4σ range %) + β × max(4σ range %)`


In [ ]:
def compute_4sigma_range_pct(
    cd_values: np.ndarray,
    ref_median: float = None,
    lower_pct: float = LOWER_PCT,
    upper_pct: float = UPPER_PCT,
) -> float:
    """4σ range % — ref_median 기준 정규화."""
    if ref_median is None:
        ref_median = float(np.median(cd_values))
    arr = np.asarray(cd_values, dtype=np.float64)
    if len(arr) < 2 or ref_median == 0.0:
        return 0.0
    cd_norm = arr / ref_median
    return (np.percentile(cd_norm, upper_pct) - np.percentile(cd_norm, lower_pct)) * 100.0


def compute_combined_4sigma_after_alignment(
    labels: np.ndarray,
    cd: np.ndarray,
    ref_median: float = None,
    lower_pct: float = LOWER_PCT,
    upper_pct: float = UPPER_PCT,
) -> float:
    """각 클러스터 median → ref_median 정렬 후 합산 분포의 4σ range %.

    OPC 레시피가 클러스터별 median CD를 ref_median으로 보정할 때
    전체 웨이퍼에 잔존하는 CD 산포를 측정합니다.
    """
    if ref_median is None:
        ref_median = OVERALL_MEDIAN_CD
    if ref_median == 0.0:
        return 0.0
    parts = []
    for lbl in np.unique(labels):
        cd_cl = cd[labels == lbl]
        if len(cd_cl) == 0:
            continue
        cl_med = float(np.median(cd_cl))
        parts.append(cd_cl - cl_med + ref_median)
    if not parts:
        return 0.0
    combined = np.concatenate(parts)
    cd_norm = combined / ref_median
    return (np.percentile(cd_norm, upper_pct) - np.percentile(cd_norm, lower_pct)) * 100.0


def compute_cluster_stats(labels: np.ndarray, cd: np.ndarray) -> dict:
    """클러스터별 CD 통계 계산."""
    unique_labels = np.unique(labels)
    stats_dict, count_dict, median_dict, sigma_nm_dict = {}, {}, {}, {}
    for lbl in unique_labels:
        mask = labels == lbl
        cd_cl = cd[mask]
        stats_dict[lbl]    = compute_4sigma_range_pct(cd_cl, ref_median=OVERALL_MEDIAN_CD)
        count_dict[lbl]    = int(mask.sum())
        median_dict[lbl]   = float(np.median(cd_cl))
        sigma_nm_dict[lbl] = float(np.std(cd_cl))

    ranges = np.array([stats_dict[l] for l in unique_labels])
    ns     = np.array([count_dict[l] for l in unique_labels])

    combined_4sigma = compute_combined_4sigma_after_alignment(labels, cd)

    return {
        'combined_4sigma_pct'    : combined_4sigma,
        'four_sigma_range_pct'   : stats_dict,
        'sigma_per_cluster'      : sigma_nm_dict,
        'median_per_cluster'     : median_dict,
        'cluster_counts'         : count_dict,
        'weighted_mean_4spct'    : float(np.average(ranges, weights=ns)),
        'unweighted_mean_4spct'  : float(ranges.mean()),
        'max_4spct'              : float(ranges.max()),
        'n_clusters'             : len(unique_labels),
        'min_count'              : int(ns.min()),
    }


def cost_function(
    labels: np.ndarray,
    cd: np.ndarray,
    min_count: int = MIN_CLUSTER_COUNT,
    cost_mode: str = None,
    lambda_penalty: float = None,
    baseline_4sigma: float = None,
    max_cluster_4sigma_threshold_ratio: float = None,
) -> float:
    """3-mode cost function.
    cost_mode: "combined" | "soft_penalty" | "hard_constraint"
    """
    _mode   = cost_mode   if cost_mode   is not None else COST_MODE
    _lambda = lambda_penalty if lambda_penalty is not None else LAMBDA_PENALTY
    _ratio  = max_cluster_4sigma_threshold_ratio if max_cluster_4sigma_threshold_ratio is not None else MAX_CLUSTER_4SIGMA_THRESHOLD_RATIO
    _base   = baseline_4sigma if baseline_4sigma is not None else BASELINE_4SIGMA_RANGE_PCT

    unique_labels, ns = np.unique(labels, return_counts=True)
    if ns.min() < min_count:
        return float('inf')

    combined = compute_combined_4sigma_after_alignment(labels, cd)

    if _mode == "combined":
        return combined

    per_cluster_4sigma = np.array([
        compute_4sigma_range_pct(cd[labels == lbl], ref_median=OVERALL_MEDIAN_CD)
        for lbl in unique_labels
    ])

    if _mode == "soft_penalty":
        weighted_mean = float(np.average(per_cluster_4sigma, weights=ns))
        return combined + _lambda * weighted_mean

    elif _mode == "hard_constraint":
        max_4sigma = float(per_cluster_4sigma.max())
        threshold  = _base * _ratio
        if max_4sigma > threshold:
            return float('inf')
        return combined

    else:
        raise ValueError(f"Unknown cost_mode: {_mode!r}")


def print_cluster_stats(stats: dict, method: str = '') -> None:
    baseline = BASELINE_4SIGMA_RANGE_PCT
    cost = stats['combined_4sigma_pct']
    imp  = (baseline - cost) / baseline * 100 if baseline > 0 else 0.0
    print(f'\n── {method} ──')
    print(f'  클러스터 수                   : {stats["n_clusters"]}')
    print(f'  최소 클러스터 크기            : {stats["min_count"]:,}')
    print(f'  기준선 4σ range %             : {baseline:.4f} %')
    print(f'  Cost ({COST_MODE})            : {cost:.4f} % (개선: {imp:.1f}%)')
    print(f'  클러스터 가중평균 4σ range %  : {stats["weighted_mean_4spct"]:.4f} %  [참고]')
    print(f'  클러스터 최대 4σ range %      : {stats["max_4spct"]:.4f} %  [참고]')


# 기준선 확인
baseline_check = compute_4sigma_range_pct(y, ref_median=OVERALL_MEDIAN_CD)
assert abs(baseline_check - BASELINE_4SIGMA_RANGE_PCT) < 1e-8, \
    f'Baseline mismatch: {baseline_check:.10f} vs {BASELINE_4SIGMA_RANGE_PCT:.10f}'
print('Cost function 정의 완료.')
print(f'  Cost = combined 4σ range % after median alignment')
print(f'  기준선 4σ range % = {BASELINE_4SIGMA_RANGE_PCT:.4f} %')


## 5. SHAP 기반 Feature Engineering

In [ ]:
# ── 5-1. XGBoost CD 예측 모델 ──────────────────────────────────
X_tr, X_val, y_tr, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=RANDOM_STATE)

xgb_model = xgb.XGBRegressor(
    n_estimators=XGB_N_ESTIMATORS, max_depth=XGB_MAX_DEPTH,
    learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
    n_jobs=-1, random_state=RANDOM_STATE, verbosity=0,
    eval_metric='rmse', early_stopping_rounds=20,
)
xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

y_pred = xgb_model.predict(X_val)
rmse = np.sqrt(np.mean((y_pred - y_val)**2))
r2   = 1 - np.sum((y_pred - y_val)**2) / np.sum((y_val - y_val.mean())**2)
print(f'XGBoost: RMSE={rmse:.3f} nm, R²={r2:.4f}')
joblib.dump(xgb_model, f'{OUTPUT_DIR}/xgb_cd_model.pkl')

In [ ]:
# ── 5-2. SHAP 계산 & Feature 선택 ─────────────────────────────
shap_idx    = np.random.RandomState(RANDOM_STATE).choice(len(X_scaled), SHAP_SAMPLE_N, replace=False)
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_scaled[shap_idx])

shap_importance = pd.Series(
    np.abs(shap_values).mean(axis=0), index=feat_cols_clean
).sort_values(ascending=False)

selected_features = shap_importance.head(SHAP_TOP_K).index.tolist()
X_sel = X_scaled_df[selected_features].values

print(f'SHAP Top-{SHAP_TOP_K} feature 선택 완료')
cumsum_pct = shap_importance.cumsum() / shap_importance.sum() * 100
print(f'Top-{SHAP_TOP_K}이 전체 SHAP 중요도의 {cumsum_pct.iloc[SHAP_TOP_K-1]:.1f}% 커버')

In [ ]:
# ── 5-3. SHAP 시각화 ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

top_feats = shap_importance.head(SHAP_TOP_K)
colors = cm.viridis(np.linspace(0.2, 0.8, SHAP_TOP_K))[::-1]
axes[0].barh(top_feats.index[::-1], top_feats.values[::-1], color=colors)
axes[0].set_title(f'SHAP Feature Importance (Top {SHAP_TOP_K})', fontweight='bold')
axes[0].set_xlabel('mean |SHAP value|')

axes[1].plot(range(1, len(cumsum_pct)+1), cumsum_pct.values, 'steelblue', lw=2)
axes[1].axhline(90, color='red', ls='--', lw=1, label='90%')
axes[1].axhline(95, color='orange', ls='--', lw=1, label='95%')
axes[1].axvline(SHAP_TOP_K, color='green', ls='--', lw=1.5, label=f'Top-{SHAP_TOP_K}')
axes[1].set_title('Cumulative SHAP Importance', fontweight='bold')
axes[1].set_xlabel('Number of Features'); axes[1].set_ylabel('Cumulative (%)')
axes[1].legend(); axes[1].set_xlim(1, len(feat_cols_clean))

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_shap_importance.png', bbox_inches='tight')
plt.show()

# Beeswarm
# selected_features는 SHAP 중요도 순으로 정렬된 상위 K개
# → 원본 feature 순서 인덱스로 슬라이싱해야 정렬 일치
sel_indices = [feat_cols_clean.index(f) for f in selected_features]
shap_exp = shap.Explanation(
    values=shap_values[:, sel_indices],
    base_values=explainer.expected_value,
    data=X_scaled[shap_idx][:, sel_indices],
    feature_names=selected_features,
)
plt.figure(figsize=(10, 7))
shap.plots.beeswarm(shap_exp, max_display=SHAP_TOP_K, show=False)
plt.title(f'SHAP Beeswarm (Top {SHAP_TOP_K})', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/03_shap_beeswarm.png', bbox_inches='tight')
plt.show()

## 6. 유틸리티 함수

In [ ]:
def merge_small_clusters(labels: np.ndarray, X: np.ndarray, min_count: int) -> np.ndarray:
    """크기 min_count 미만 클러스터를 centroid KNN으로 가장 가까운 클러스터에 반복 병합."""
    labels = labels.copy()
    for _ in range(1000):
        counts = Counter(labels)
        small  = sorted([l for l, c in counts.items() if c < min_count], key=lambda l: counts[l])
        if not small:
            break
        tgt   = small[0]
        valid = [l for l in np.unique(labels) if l != tgt]
        if not valid:
            break
        mask = labels == tgt
        centroid = X[mask].mean(axis=0, keepdims=True)
        valid_centroids = np.array([X[labels == l].mean(axis=0) for l in valid])
        nearest = valid[np.linalg.norm(valid_centroids - centroid, axis=1).argmin()]
        labels[mask] = nearest
    return labels


def relabel_sequential(labels: np.ndarray) -> np.ndarray:
    mapping = {old: new for new, old in enumerate(sorted(np.unique(labels)))}
    return np.vectorize(mapping.get)(labels)


def run_grid_search(label_fn, param_range, desc):
    """Grid search 공통 루틴. label_fn(param) → raw_labels."""
    results = []
    for param in tqdm(param_range, desc=desc):
        raw     = label_fn(param)
        merged  = merge_small_clusters(raw, X_sel, MIN_CLUSTER_COUNT)
        final   = relabel_sequential(merged)
        stats   = compute_cluster_stats(final, y)
        cost    = cost_function(final, y)
        results.append({'param': param, 'labels': final, 'stats': stats, 'cost': cost})
    return results




def run_optuna_study(objective_fn, n_trials: int, n_jobs: int = 1, direction: str = "minimize"):
    """Optuna study 실행 헬퍼."""
    study = optuna.create_study(direction=direction)
    study.optimize(objective_fn, n_trials=n_trials, n_jobs=n_jobs, show_progress_bar=False)
    return study


def plot_alignment_result(labels: np.ndarray, y: np.ndarray, method_name: str) -> None:
    """클러스터 median alignment 전/후 분포 시각화 + 개선율 출력."""
    ref = OVERALL_MEDIAN_CD
    baseline = BASELINE_4SIGMA_RANGE_PCT

    # Aligned distribution
    parts = []
    for lbl in np.unique(labels):
        cd_cl = y[labels == lbl]
        parts.append(cd_cl - float(np.median(cd_cl)) + ref)
    aligned = np.concatenate(parts)
    cost = compute_combined_4sigma_after_alignment(labels, y)
    improvement = (baseline - cost) / baseline * 100 if baseline > 0 else 0.0

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # 왼쪽: before/after 히스토그램
    bins = np.linspace(min(y.min(), aligned.min()), max(y.max(), aligned.max()), 120)
    axes[0].hist(y, bins=bins, alpha=0.45, label=f'분리 전 ({baseline:.2f}%)', color='steelblue', density=True)
    axes[0].hist(aligned, bins=bins, alpha=0.55, label=f'정렬 후 ({cost:.2f}%)', color='tomato', density=True)
    axes[0].axvline(ref, color='black', linestyle='--', linewidth=1, label=f'ref={ref:.1f}nm')
    axes[0].set_title(f'{method_name} — Before / After Alignment')
    axes[0].set_xlabel('CD (nm)')
    axes[0].set_ylabel('density')
    axes[0].legend(fontsize=9)

    # 오른쪽: 클러스터별 box plot (median 순 정렬, 최대 40개)
    unique_lbls = np.unique(labels)
    sorted_lbls = sorted(unique_lbls, key=lambda l: float(np.median(y[labels == l])))
    display_lbls = sorted_lbls[:40]
    data_box = [y[labels == l] for l in display_lbls]
    axes[1].boxplot(data_box, notch=False, sym='', widths=0.6,
                    medianprops=dict(color='tomato', linewidth=2))
    axes[1].axhline(ref, color='navy', linestyle='--', linewidth=1,
                    label=f'overall median={ref:.1f}nm')
    n_display = len(display_lbls)
    axes[1].set_title(f'{method_name} — 클러스터별 CD (k={len(unique_lbls)}, 표시:{n_display})')
    axes[1].set_xlabel('클러스터 (median 순)')
    axes[1].set_ylabel('CD (nm)')
    axes[1].legend(fontsize=9)
    axes[1].set_xticks([])

    plt.suptitle(f'[{method_name}]  개선율: {improvement:+.2f}%  ({baseline:.4f}% → {cost:.4f}%)',
                 fontweight='bold', fontsize=11)
    plt.tight_layout()
    plt.show()
    print(f'  ✓ {method_name}: 분리 전 {baseline:.4f}%  →  정렬 후 {cost:.4f}%  (개선: {improvement:+.2f}%)')


print('유틸리티 함수 정의 완료.')

## 7. Method 1 — Decision Tree Clustering

In [ ]:
# ── Method 1: Decision Tree — Optuna 최적화 ─────────────────────
from sklearn.tree import DecisionTreeRegressor

def _dt_objective(trial):
    max_leaf_nodes   = trial.suggest_int("max_leaf_nodes",   5, 150)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 10, 500)
    dt = DecisionTreeRegressor(
        max_leaf_nodes=max_leaf_nodes,
        min_samples_leaf=min_samples_leaf,
        random_state=RANDOM_STATE,
    )
    dt.fit(X_sel, y)
    raw = dt.apply(X_sel)
    labels = relabel_sequential(merge_small_clusters(raw, X_sel, MIN_CLUSTER_COUNT))
    return cost_function(labels, y)

print(f'Decision Tree Optuna ({OPTUNA_N_TRIALS} trials, n_jobs={OPTUNA_N_JOBS})')
dt_study = run_optuna_study(_dt_objective, OPTUNA_N_TRIALS, OPTUNA_N_JOBS)

# 최적 파라미터로 레이블 재생성
best_dt_param = dt_study.best_trial.params
dt_best = DecisionTreeRegressor(
    max_leaf_nodes=best_dt_param["max_leaf_nodes"],
    min_samples_leaf=best_dt_param["min_samples_leaf"],
    random_state=RANDOM_STATE,
)
dt_best.fit(X_sel, y)
raw = dt_best.apply(X_sel)
best_dt_labels = relabel_sequential(merge_small_clusters(raw, X_sel, MIN_CLUSTER_COUNT))
best_dt_stats  = compute_cluster_stats(best_dt_labels, y)
best_dt_cost   = cost_function(best_dt_labels, y)

print(f'  최적 파라미터: max_leaf_nodes={best_dt_param["max_leaf_nodes"]}, min_samples_leaf={best_dt_param["min_samples_leaf"]}')
print_cluster_stats(best_dt_stats, 'Decision Tree')
plot_alignment_result(best_dt_labels, y, 'Decision Tree')

## 8. Method 2 — K-Means Clustering

In [ ]:
# ── Method 2: K-Means — Optuna 최적화 ──────────────────────────
from sklearn.cluster import MiniBatchKMeans

def _km_objective(trial):
    n_clusters = trial.suggest_int("n_clusters", 5, 100)
    n_init     = trial.suggest_int("n_init", 3, 10)
    km = MiniBatchKMeans(n_clusters=n_clusters, batch_size=10_000,
                         n_init=n_init, random_state=RANDOM_STATE)
    raw = km.fit_predict(X_sel)
    labels = relabel_sequential(merge_small_clusters(raw, X_sel, MIN_CLUSTER_COUNT))
    return cost_function(labels, y)

print(f'K-Means Optuna ({OPTUNA_N_TRIALS} trials, n_jobs={OPTUNA_N_JOBS})')
km_study = run_optuna_study(_km_objective, OPTUNA_N_TRIALS, OPTUNA_N_JOBS)

best_km_param = km_study.best_trial.params
km_best = MiniBatchKMeans(
    n_clusters=best_km_param["n_clusters"],
    batch_size=10_000,
    n_init=best_km_param["n_init"],
    random_state=RANDOM_STATE,
)
raw = km_best.fit_predict(X_sel)
best_km_labels = relabel_sequential(merge_small_clusters(raw, X_sel, MIN_CLUSTER_COUNT))
best_km_stats  = compute_cluster_stats(best_km_labels, y)
best_km_cost   = cost_function(best_km_labels, y)

print(f'  최적 파라미터: n_clusters={best_km_param["n_clusters"]}, n_init={best_km_param["n_init"]}')
print_cluster_stats(best_km_stats, 'K-Means')
plot_alignment_result(best_km_labels, y, 'K-Means')

## 9. Method 3 — Autoencoder + K-Means (DL)

In [ ]:
# ── 9-1. Autoencoder 아키텍처 ──────────────────────────────────
class LayoutAutoencoder(nn.Module):
    """FC Autoencoder for layout feature embedding.
    
    Encoder: input → 256 → 128 → 64 → latent
    Decoder: latent → 64 → 128 → 256 → input
    BatchNorm + LeakyReLU + Dropout for robustness.
    """
    def __init__(self, input_dim: int, latent_dim: int, dropout: float = 0.1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.1), nn.Dropout(dropout),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.LeakyReLU(0.1), nn.Dropout(dropout),
            nn.Linear(128, 64),        nn.BatchNorm1d(64),  nn.LeakyReLU(0.1),
            nn.Linear(64, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.BatchNorm1d(64),  nn.LeakyReLU(0.1),
            nn.Linear(64, 128),        nn.BatchNorm1d(128), nn.LeakyReLU(0.1), nn.Dropout(dropout),
            nn.Linear(128, 256),       nn.BatchNorm1d(256), nn.LeakyReLU(0.1), nn.Dropout(dropout),
            nn.Linear(256, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

    def encode(self, x):
        return self.encoder(x)


INPUT_DIM = X_sel.shape[1]
ae_model  = LayoutAutoencoder(INPUT_DIM, AE_LATENT_DIM).to(DEVICE)
total_params = sum(p.numel() for p in ae_model.parameters())
print(f'Autoencoder 초기화 완료')
print(f'  Input dim    : {INPUT_DIM}')
print(f'  Latent dim   : {AE_LATENT_DIM}')
print(f'  Total params : {total_params:,}')
print(ae_model)

In [ ]:
# ── 9-2. Autoencoder 학습 ──────────────────────────────────────
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

X_tensor = torch.tensor(X_sel, dtype=torch.float32)
dataset  = TensorDataset(X_tensor)
loader   = DataLoader(dataset, batch_size=AE_BATCH_SIZE, shuffle=True,
                       num_workers=0, pin_memory=(DEVICE.type == 'cuda'))

optimizer = torch.optim.Adam(ae_model.parameters(), lr=AE_LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
criterion = nn.MSELoss()

train_losses, best_loss, patience_cnt = [], float('inf'), 0

print(f'Autoencoder 학습 시작 (epochs={AE_EPOCHS}, batch={AE_BATCH_SIZE}, device={DEVICE})')
for epoch in range(1, AE_EPOCHS + 1):
    ae_model.train()
    epoch_loss = 0.0
    for (xb,) in loader:
        xb = xb.to(DEVICE)
        recon, _ = ae_model(xb)
        loss = criterion(recon, xb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item() * len(xb)
    epoch_loss /= len(dataset)
    train_losses.append(epoch_loss)
    scheduler.step(epoch_loss)

    if epoch_loss < best_loss:
        best_loss    = epoch_loss
        patience_cnt = 0
        torch.save(ae_model.state_dict(), f'{OUTPUT_DIR}/ae_best.pt')
    else:
        patience_cnt += 1

    if epoch % 5 == 0 or epoch == 1:
        print(f'  Epoch {epoch:3d}/{AE_EPOCHS}  loss={epoch_loss:.6f}  lr={optimizer.param_groups[0]["lr"]:.2e}')

    if patience_cnt >= AE_PATIENCE:
        print(f'  Early stopping at epoch {epoch} (patience={AE_PATIENCE})')
        break

# 학습 곡선
plt.figure(figsize=(8, 4))
plt.plot(train_losses, 'steelblue', lw=2)
plt.title('Autoencoder Training Loss (MSE)', fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/06_ae_training_loss.png', bbox_inches='tight')
plt.show()
print(f'\n최적 loss: {best_loss:.6f}')

In [ ]:
# ── 9-3. Latent 표현 추출 ─────────────────────────────────────
ae_model.load_state_dict(torch.load(f'{OUTPUT_DIR}/ae_best.pt', map_location=DEVICE))
ae_model.eval()

latent_list = []
with torch.no_grad():
    for i in range(0, len(X_tensor), AE_BATCH_SIZE * 4):
        xb = X_tensor[i:i + AE_BATCH_SIZE * 4].to(DEVICE)
        latent_list.append(ae_model.encode(xb).cpu().numpy())
X_latent = np.concatenate(latent_list, axis=0)

print(f'Latent 추출 완료: {X_latent.shape}')
print(f'  Latent 표현 통계: mean={X_latent.mean():.4f}, std={X_latent.std():.4f}')

# 재구성 오류 (품질 확인)
with torch.no_grad():
    sample_idx = np.random.choice(len(X_tensor), 5000)
    xb_sample  = X_tensor[sample_idx].to(DEVICE)
    recon, _   = ae_model(xb_sample)
    recon_err  = criterion(recon, xb_sample).item()
print(f'  재구성 MSE (5K 샘플): {recon_err:.6f}')

In [ ]:
# ── 9-4. AE Latent + K-Means — Optuna 최적화 ──────────────────
def _ae_km_objective(trial):
    n_clusters = trial.suggest_int("n_clusters", 5, 60)
    n_init     = trial.suggest_int("n_init", 3, 10)
    km = MiniBatchKMeans(n_clusters=n_clusters, batch_size=10_000,
                         n_init=n_init, random_state=RANDOM_STATE)
    raw = km.fit_predict(X_latent)
    labels = relabel_sequential(merge_small_clusters(raw, X_latent, MIN_CLUSTER_COUNT))
    return cost_function(labels, y)

print(f'AE+KMeans Optuna ({OPTUNA_N_TRIALS} trials, n_jobs={OPTUNA_N_JOBS})')
ae_study = run_optuna_study(_ae_km_objective, OPTUNA_N_TRIALS, OPTUNA_N_JOBS)

best_ae_param = ae_study.best_trial.params
km_ae_best = MiniBatchKMeans(
    n_clusters=best_ae_param["n_clusters"],
    batch_size=10_000,
    n_init=best_ae_param["n_init"],
    random_state=RANDOM_STATE,
)
raw = km_ae_best.fit_predict(X_latent)
best_ae_labels = relabel_sequential(merge_small_clusters(raw, X_latent, MIN_CLUSTER_COUNT))
best_ae_stats  = compute_cluster_stats(best_ae_labels, y)
best_ae_cost   = cost_function(best_ae_labels, y)

print(f'  최적 파라미터: n_clusters={best_ae_param["n_clusters"]}, n_init={best_ae_param["n_init"]}')
print_cluster_stats(best_ae_stats, 'AE+KMeans')
plot_alignment_result(best_ae_labels, y, 'Autoencoder + KMeans')

## 10. Method 4 — GMM (Gaussian Mixture Model)

**특징**: 각 클러스터를 다변량 정규분포로 모델링. 비구형/타원형 클러스터 탐지에 강점.  
**핵심 원리**: EM 알고리즘으로 사후 확률(soft-assignment) 최대화.  
**제약 처리**: 전체 데이터(64K)에 대해 50K subsample로 fit, 나머지는 `predict()` 할당 후 소규모 클러스터 병합.  
**trade-off**: K-Means보다 유연하지만 covariance 행렬 크기로 학습 속도 ↑.


In [ ]:
# ── Method 4: GMM — Optuna 최적화 ──────────────────────────────
from sklearn.mixture import GaussianMixture

GMM_SUBSAMPLE = 50_000

def _gmm_objective(trial):
    n_components    = trial.suggest_int("n_components", 5, 60)
    covariance_type = trial.suggest_categorical("covariance_type", ["full", "tied", "diag"])
    rng_idx = np.random.RandomState(RANDOM_STATE).choice(
        len(X_sel), min(GMM_SUBSAMPLE, len(X_sel)), replace=False
    )
    gm = GaussianMixture(n_components=n_components, covariance_type=covariance_type,
                         n_init=3, random_state=RANDOM_STATE, max_iter=200)
    gm.fit(X_sel[rng_idx])
    raw = gm.predict(X_sel)
    labels = relabel_sequential(merge_small_clusters(raw, X_sel, MIN_CLUSTER_COUNT))
    return cost_function(labels, y)

print(f'GMM Optuna ({OPTUNA_N_TRIALS} trials, n_jobs={OPTUNA_N_JOBS})')
gmm_study = run_optuna_study(_gmm_objective, OPTUNA_N_TRIALS, OPTUNA_N_JOBS)

best_gmm_param = gmm_study.best_trial.params
rng_idx_best = np.random.RandomState(RANDOM_STATE).choice(
    len(X_sel), min(GMM_SUBSAMPLE, len(X_sel)), replace=False
)
gm_best = GaussianMixture(
    n_components=best_gmm_param["n_components"],
    covariance_type=best_gmm_param["covariance_type"],
    n_init=3, random_state=RANDOM_STATE, max_iter=200
)
gm_best.fit(X_sel[rng_idx_best])
raw = gm_best.predict(X_sel)
best_gmm_labels = relabel_sequential(merge_small_clusters(raw, X_sel, MIN_CLUSTER_COUNT))
best_gmm_stats  = compute_cluster_stats(best_gmm_labels, y)
best_gmm_cost   = cost_function(best_gmm_labels, y)

print(f'  최적 파라미터: n_components={best_gmm_param["n_components"]}, covariance_type={best_gmm_param["covariance_type"]}')
print_cluster_stats(best_gmm_stats, 'GMM')
plot_alignment_result(best_gmm_labels, y, 'GMM')

## 11. Method 5 — Bisecting K-Means

**특징**: 계층적 분할 방식의 K-Means. 각 iteration에서 내부 분산이 가장 큰 클러스터를 이분할.  
**핵심 원리**: 전체 → 2개 → 4개 → ... 로 이진 분할. top-down 계층 구조.  
**장점**: 일반 K-Means 대비 balanced cluster 경향, 빠른 수렴.  
**적용**: sklearn `BisectingKMeans`로 64K 전체 직접 fit.


In [ ]:
# ── Method 5: Bisecting K-Means — Optuna 최적화 ─────────────────
from sklearn.cluster import BisectingKMeans

def _bkm_objective(trial):
    n_clusters    = trial.suggest_int("n_clusters", 5, 100)
    bisecting_strategy = trial.suggest_categorical("bisecting_strategy", ["biggest_inertia", "largest_cluster"])
    bkm = BisectingKMeans(n_clusters=n_clusters, n_init=3,
                          bisecting_strategy=bisecting_strategy,
                          random_state=RANDOM_STATE)
    raw = bkm.fit_predict(X_sel)
    labels = relabel_sequential(merge_small_clusters(raw, X_sel, MIN_CLUSTER_COUNT))
    return cost_function(labels, y)

print(f'Bisecting K-Means Optuna ({OPTUNA_N_TRIALS} trials, n_jobs={OPTUNA_N_JOBS})')
bkm_study = run_optuna_study(_bkm_objective, OPTUNA_N_TRIALS, OPTUNA_N_JOBS)

best_bkm_param = bkm_study.best_trial.params
bkm_best = BisectingKMeans(
    n_clusters=best_bkm_param["n_clusters"],
    bisecting_strategy=best_bkm_param["bisecting_strategy"],
    n_init=3, random_state=RANDOM_STATE
)
raw = bkm_best.fit_predict(X_sel)
best_bkm_labels = relabel_sequential(merge_small_clusters(raw, X_sel, MIN_CLUSTER_COUNT))
best_bkm_stats  = compute_cluster_stats(best_bkm_labels, y)
best_bkm_cost   = cost_function(best_bkm_labels, y)

print(f'  최적 파라미터: n_clusters={best_bkm_param["n_clusters"]}, strategy={best_bkm_param["bisecting_strategy"]}')
print_cluster_stats(best_bkm_stats, 'Bisecting K-Means')
plot_alignment_result(best_bkm_labels, y, 'Bisecting K-Means')

## 12. Method 6 — Agglomerative Clustering (Ward Linkage)

**특징**: 상향식(bottom-up) 계층적 군집화. 각 포인트를 개별 클러스터로 시작해 Ward 기준으로 병합.  
**핵심 원리**: Ward linkage = 병합 후 클러스터 내 분산 증가량 최소화 → CD 산포 최소화 목적에 부합.  
**스케일 전략**: 64K 전체 fit 불가 → 30K subsample `AgglomerativeClustering` fit 후 KNN(k=5)으로 나머지 할당.  
**장점**: 노이즈에 강건, 비구형 클러스터 탐지.


In [ ]:
# ── Method 6: Agglomerative Ward — Optuna 최적화 ────────────────
from sklearn.cluster import AgglomerativeClustering
from sklearn.neighbors import KNeighborsClassifier

AGG_SUBSAMPLE = 30_000

def _agg_objective(trial):
    n_clusters = trial.suggest_int("n_clusters", 5, 80)
    sub_idx = np.random.RandomState(RANDOM_STATE).choice(
        len(X_sel), min(AGG_SUBSAMPLE, len(X_sel)), replace=False
    )
    X_sub = X_sel[sub_idx]
    agg = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
    sub_labels = agg.fit_predict(X_sub)
    knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
    knn.fit(X_sub, sub_labels)
    raw = knn.predict(X_sel)
    labels = relabel_sequential(merge_small_clusters(raw, X_sel, MIN_CLUSTER_COUNT))
    return cost_function(labels, y)

print(f'Agglomerative Ward Optuna ({OPTUNA_N_TRIALS} trials, n_jobs={OPTUNA_N_JOBS})')
agg_study = run_optuna_study(_agg_objective, OPTUNA_N_TRIALS, OPTUNA_N_JOBS)

best_agg_param = agg_study.best_trial.params
sub_idx_best = np.random.RandomState(RANDOM_STATE).choice(
    len(X_sel), min(AGG_SUBSAMPLE, len(X_sel)), replace=False
)
X_sub_best = X_sel[sub_idx_best]
agg_best = AgglomerativeClustering(n_clusters=best_agg_param["n_clusters"], linkage='ward')
sub_labels_best = agg_best.fit_predict(X_sub_best)
knn_best = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
knn_best.fit(X_sub_best, sub_labels_best)
raw = knn_best.predict(X_sel)
best_agg_labels = relabel_sequential(merge_small_clusters(raw, X_sel, MIN_CLUSTER_COUNT))
best_agg_stats  = compute_cluster_stats(best_agg_labels, y)
best_agg_cost   = cost_function(best_agg_labels, y)

print(f'  최적 파라미터: n_clusters={best_agg_param["n_clusters"]}')
print_cluster_stats(best_agg_stats, 'Agglomerative Ward')
plot_alignment_result(best_agg_labels, y, 'Agglomerative Ward')

## 13. Method 7 — HDBSCAN (Hierarchical Density-Based Clustering)

**특징**: 밀도 기반 클러스터링. 밀도가 낮은 영역을 noise로 처리하며 클러스터 수를 자동 결정.  
**핵심 원리**: 각 포인트의 core distance → minimum spanning tree → 계층적 클러스터 추출.  
**noise 처리**: 할당 안 된 포인트(-1) → 인접 클러스터에 KNN으로 강제 할당.  
**환경 호환**: sklearn 1.3+ `HDBSCAN` 내장 사용, 없으면 `hdbscan` 패키지 fallback.


In [ ]:
# ── Method 7: HDBSCAN — Optuna 최적화 ──────────────────────────
try:
    from sklearn.cluster import HDBSCAN as HDBSCAN_cls
    _hdbscan_n_jobs_key = "n_jobs"
except ImportError:
    import hdbscan as _hdbscan_pkg
    HDBSCAN_cls = _hdbscan_pkg.HDBSCAN
    _hdbscan_n_jobs_key = "core_dist_n_jobs"

def _hdb_objective(trial):
    min_cluster_size = trial.suggest_int("min_cluster_size", MIN_CLUSTER_COUNT, MIN_CLUSTER_COUNT * 6)
    min_samples      = trial.suggest_int("min_samples", 5, 50)
    kwargs = {_hdbscan_n_jobs_key: -1}
    hdb = HDBSCAN_cls(min_cluster_size=min_cluster_size, min_samples=min_samples, **kwargs)
    raw = hdb.fit_predict(X_sel)
    # noise(-1) 처리
    noise_mask = raw == -1
    if noise_mask.sum() > 0:
        valid_mask = ~noise_mask
        if valid_mask.sum() < MIN_CLUSTER_COUNT:
            return float('inf')
        knn_hdb = KNeighborsClassifier(n_neighbors=3, n_jobs=-1)
        knn_hdb.fit(X_sel[valid_mask], raw[valid_mask])
        raw[noise_mask] = knn_hdb.predict(X_sel[noise_mask])
    labels = relabel_sequential(merge_small_clusters(raw, X_sel, MIN_CLUSTER_COUNT))
    return cost_function(labels, y)

print(f'HDBSCAN Optuna ({OPTUNA_N_TRIALS} trials, n_jobs={OPTUNA_N_JOBS})')
hdb_study = run_optuna_study(_hdb_objective, OPTUNA_N_TRIALS, OPTUNA_N_JOBS)

best_hdb_param = hdb_study.best_trial.params
kwargs_best = {_hdbscan_n_jobs_key: -1}
hdb_best = HDBSCAN_cls(
    min_cluster_size=best_hdb_param["min_cluster_size"],
    min_samples=best_hdb_param["min_samples"],
    **kwargs_best
)
raw = hdb_best.fit_predict(X_sel)
noise_mask = raw == -1
if noise_mask.sum() > 0:
    valid_mask = ~noise_mask
    knn_hdb = KNeighborsClassifier(n_neighbors=3, n_jobs=-1)
    knn_hdb.fit(X_sel[valid_mask], raw[valid_mask])
    raw[noise_mask] = knn_hdb.predict(X_sel[noise_mask])
best_hdb_labels = relabel_sequential(merge_small_clusters(raw, X_sel, MIN_CLUSTER_COUNT))
best_hdb_stats  = compute_cluster_stats(best_hdb_labels, y)
best_hdb_cost   = cost_function(best_hdb_labels, y)

print(f'  최적 파라미터: min_cluster_size={best_hdb_param["min_cluster_size"]}, min_samples={best_hdb_param["min_samples"]}')
print_cluster_stats(best_hdb_stats, 'HDBSCAN')
plot_alignment_result(best_hdb_labels, y, 'HDBSCAN')

## 14. Method 8 — Spectral Clustering

**특징**: 유사도 그래프의 Laplacian eigenvector 기반 임베딩 후 K-Means. 비볼록 클러스터에 강점.  
**핵심 원리**: feature 공간 거리가 아닌 그래프 연결성(topological) 기반 분리.  
**스케일 전략**: 그래프 행렬이 O(n²) 메모리 → 8K subsample spectral fit + KNN으로 64K 전체 할당.  
**주의**: 다른 방법 대비 느림. subsample 크기가 결과 품질에 영향.


In [ ]:
# ── Method 8: Spectral Clustering — Optuna 최적화 ───────────────
from sklearn.cluster import SpectralClustering
from sklearn.neighbors import KNeighborsClassifier

SPEC_SUBSAMPLE = 8_000

def _spec_objective(trial):
    n_clusters  = trial.suggest_int("n_clusters", 5, 30)
    n_neighbors = trial.suggest_int("n_neighbors", 5, 20)
    sub_idx = np.random.RandomState(RANDOM_STATE).choice(
        len(X_sel), min(SPEC_SUBSAMPLE, len(X_sel)), replace=False
    )
    X_sub = X_sel[sub_idx]
    sc = SpectralClustering(n_clusters=n_clusters, affinity='nearest_neighbors',
                             n_neighbors=n_neighbors, assign_labels='kmeans',
                             n_jobs=-1, random_state=RANDOM_STATE)
    sub_labels = sc.fit_predict(X_sub)
    knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
    knn.fit(X_sub, sub_labels)
    raw = knn.predict(X_sel)
    labels = relabel_sequential(merge_small_clusters(raw, X_sel, MIN_CLUSTER_COUNT))
    return cost_function(labels, y)

print(f'Spectral Optuna ({OPTUNA_N_TRIALS} trials, n_jobs={OPTUNA_N_JOBS})')
spec_study = run_optuna_study(_spec_objective, OPTUNA_N_TRIALS, OPTUNA_N_JOBS)

best_spec_param = spec_study.best_trial.params
sub_idx_best = np.random.RandomState(RANDOM_STATE).choice(
    len(X_sel), min(SPEC_SUBSAMPLE, len(X_sel)), replace=False
)
X_sub_best = X_sel[sub_idx_best]
sc_best = SpectralClustering(
    n_clusters=best_spec_param["n_clusters"],
    n_neighbors=best_spec_param["n_neighbors"],
    affinity='nearest_neighbors', assign_labels='kmeans',
    n_jobs=-1, random_state=RANDOM_STATE
)
sub_labels_best = sc_best.fit_predict(X_sub_best)
knn_spec = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
knn_spec.fit(X_sub_best, sub_labels_best)
raw = knn_spec.predict(X_sel)
best_spec_labels = relabel_sequential(merge_small_clusters(raw, X_sel, MIN_CLUSTER_COUNT))
best_spec_stats  = compute_cluster_stats(best_spec_labels, y)
best_spec_cost   = cost_function(best_spec_labels, y)

print(f'  최적 파라미터: n_clusters={best_spec_param["n_clusters"]}, n_neighbors={best_spec_param["n_neighbors"]}')
print_cluster_stats(best_spec_stats, 'Spectral')
plot_alignment_result(best_spec_labels, y, 'Spectral Clustering')

## 15. Method 9 — Isolation Forest + K-Means (Outlier-Robust)

**특징**: 이상치 제거 후 클러스터링. layout data의 outlier가 CD 산포를 왜곡하는 문제 해결.  
**핵심 원리**: IsolationForest로 contamination=2% 이상치 식별 → 정상 포인트에만 K-Means 적용.  
**outlier 처리**: 이상치 포인트는 가장 가까운 centroid로 사후 할당 (분석 누락 방지).  
**장점**: 공정 이상(장비 drift, 측정 오류) 영향을 클러스터 구조에서 분리.


In [ ]:
# ── Method 9: IsolationForest + K-Means — Optuna 최적화 ─────────
from sklearn.ensemble import IsolationForest
from sklearn.cluster import MiniBatchKMeans
from sklearn.neighbors import KNeighborsClassifier

def _iso_objective(trial):
    n_clusters    = trial.suggest_int("n_clusters", 5, 80)
    contamination = trial.suggest_float("contamination", 0.01, 0.1)
    n_init        = trial.suggest_int("n_init", 3, 10)

    iso = IsolationForest(contamination=contamination, n_jobs=-1, random_state=RANDOM_STATE)
    inlier_mask = iso.fit_predict(X_sel) == 1
    if inlier_mask.sum() < n_clusters * MIN_CLUSTER_COUNT:
        return float('inf')

    X_inlier = X_sel[inlier_mask]
    km = MiniBatchKMeans(n_clusters=n_clusters, batch_size=10_000,
                         n_init=n_init, random_state=RANDOM_STATE)
    inlier_labels = km.fit_predict(X_inlier)

    # outlier → nearest centroid
    raw = np.empty(len(X_sel), dtype=int)
    raw[inlier_mask] = inlier_labels
    if (~inlier_mask).sum() > 0:
        knn = KNeighborsClassifier(n_neighbors=3, n_jobs=-1)
        knn.fit(X_inlier, inlier_labels)
        raw[~inlier_mask] = knn.predict(X_sel[~inlier_mask])

    labels = relabel_sequential(merge_small_clusters(raw, X_sel, MIN_CLUSTER_COUNT))
    return cost_function(labels, y)

print(f'IsoForest+KMeans Optuna ({OPTUNA_N_TRIALS} trials, n_jobs={OPTUNA_N_JOBS})')
iso_study = run_optuna_study(_iso_objective, OPTUNA_N_TRIALS, OPTUNA_N_JOBS)

best_iso_param = iso_study.best_trial.params
iso_best = IsolationForest(contamination=best_iso_param["contamination"],
                           n_jobs=-1, random_state=RANDOM_STATE)
inlier_mask_best = iso_best.fit_predict(X_sel) == 1
X_inlier_best = X_sel[inlier_mask_best]
km_iso_best = MiniBatchKMeans(
    n_clusters=best_iso_param["n_clusters"],
    batch_size=10_000,
    n_init=best_iso_param["n_init"],
    random_state=RANDOM_STATE,
)
inlier_labels_best = km_iso_best.fit_predict(X_inlier_best)
raw_best = np.empty(len(X_sel), dtype=int)
raw_best[inlier_mask_best] = inlier_labels_best
if (~inlier_mask_best).sum() > 0:
    knn_iso = KNeighborsClassifier(n_neighbors=3, n_jobs=-1)
    knn_iso.fit(X_inlier_best, inlier_labels_best)
    raw_best[~inlier_mask_best] = knn_iso.predict(X_sel[~inlier_mask_best])
best_iso_labels = relabel_sequential(merge_small_clusters(raw_best, X_sel, MIN_CLUSTER_COUNT))
best_iso_stats  = compute_cluster_stats(best_iso_labels, y)
best_iso_cost   = cost_function(best_iso_labels, y)

print(f'  최적 파라미터: n_clusters={best_iso_param["n_clusters"]}, contamination={best_iso_param["contamination"]:.4f}, n_init={best_iso_param["n_init"]}')
print_cluster_stats(best_iso_stats, 'IsoForest+KMeans')
plot_alignment_result(best_iso_labels, y, 'IsolationForest + KMeans')

## 16. Method 10 — VAE + K-Means (Variational Autoencoder, DL)

**특징**: Variational Autoencoder로 구조화된 latent space 학습 후 K-Means.  
**핵심 원리**: AE와 달리 잠재 공간을 N(μ, σ²)으로 정규화 → 더 smooth하고 연속적인 표현.  
**clustering 사용**: inference 시 확정적 μ(mean)만 사용 → 재현 가능한 클러스터링.  
**β-VAE**: β값으로 재구성 품질 vs 잠재 공간 구조화 tradeoff 조절.  
**AE 대비 장점**: disentangled representation → 유사한 layout feature가 latent 공간에서 더 뭉침.


In [ ]:
# VAE: reparameterization trick으로 더 smooth한 잠재 표현 학습
# AE보다 구조화된 latent space → clustering에 유리

class LayoutVAE(nn.Module):
    """Variational Autoencoder for layout features.
    Encoder: input → hidden → μ, logσ²
    Decoder: z ~ N(μ, σ²) → hidden → output
    Loss: MSE recon + β·KLD
    """
    def __init__(self, input_dim: int, latent_dim: int, beta: float = 1.0, dropout: float = 0.1):
        super().__init__()
        self.beta = beta
        self.encoder_net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.1), nn.Dropout(dropout),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.LeakyReLU(0.1),
        )
        self.fc_mu     = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        self.decoder   = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.BatchNorm1d(128), nn.LeakyReLU(0.1), nn.Dropout(dropout),
            nn.Linear(128, 256),        nn.BatchNorm1d(256), nn.LeakyReLU(0.1),
            nn.Linear(256, input_dim),
        )

    def encode(self, x):
        h = self.encoder_net(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        if self.training:
            std = (0.5 * logvar).exp()
            return mu + std * torch.randn_like(std)
        return mu   # inference: deterministic (use mean)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z     = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar

    def get_latent(self, x):
        mu, _ = self.encode(x)
        return mu   # clustering에는 μ 사용 (확정적)


VAE_LATENT_DIM = AE_LATENT_DIM
VAE_BETA       = 1.0
VAE_EPOCHS     = AE_EPOCHS
VAE_PATIENCE   = AE_PATIENCE

torch.manual_seed(RANDOM_STATE)
vae_model = LayoutVAE(INPUT_DIM, VAE_LATENT_DIM, beta=VAE_BETA).to(DEVICE)
vae_opt   = torch.optim.Adam(vae_model.parameters(), lr=AE_LR, weight_decay=1e-5)
vae_sch   = torch.optim.lr_scheduler.ReduceLROnPlateau(vae_opt, patience=3, factor=0.5)

def vae_loss(recon, x, mu, logvar, beta=VAE_BETA):
    recon_loss = nn.functional.mse_loss(recon, x, reduction='sum') / len(x)
    kld        = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum() / len(x)
    return recon_loss + beta * kld

print(f'VAE 학습 시작 (latent={VAE_LATENT_DIM}, β={VAE_BETA})')
vae_losses, vae_best, vae_patience_cnt = [], float('inf'), 0
for epoch in range(1, VAE_EPOCHS + 1):
    vae_model.train(); epoch_loss = 0.0
    for (xb,) in loader:
        xb = xb.to(DEVICE)
        recon, mu, logvar = vae_model(xb)
        loss = vae_loss(recon, xb, mu, logvar)
        vae_opt.zero_grad(); loss.backward(); vae_opt.step()
        epoch_loss += loss.item() * len(xb)
    epoch_loss /= len(dataset); vae_losses.append(epoch_loss)
    vae_sch.step(epoch_loss)
    if epoch_loss < vae_best:
        vae_best = epoch_loss; vae_patience_cnt = 0
        torch.save(vae_model.state_dict(), f'{OUTPUT_DIR}/vae_best.pt')
    else:
        vae_patience_cnt += 1
    if epoch % 5 == 0 or epoch == 1:
        print(f'  Epoch {epoch:3d}  loss={epoch_loss:.6f}')
    if vae_patience_cnt >= VAE_PATIENCE:
        print(f'  Early stopping at epoch {epoch}'); break

plt.figure(figsize=(8,3))
plt.plot(vae_losses, 'mediumpurple', lw=2)
plt.title('VAE Training Loss', fontweight='bold'); plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/vae_training_loss.png', bbox_inches='tight'); plt.show()
print(f'VAE best loss: {vae_best:.6f}')


In [ ]:
# ── VAE Latent + K-Means — Optuna 최적화 ────────────────────────
def _vae_km_objective(trial):
    n_clusters = trial.suggest_int("n_clusters", 5, 60)
    n_init     = trial.suggest_int("n_init", 3, 10)
    km = MiniBatchKMeans(n_clusters=n_clusters, batch_size=10_000,
                         n_init=n_init, random_state=RANDOM_STATE)
    raw = km.fit_predict(X_vae_latent)
    labels = relabel_sequential(merge_small_clusters(raw, X_vae_latent, MIN_CLUSTER_COUNT))
    return cost_function(labels, y)

print(f'VAE+KMeans Optuna ({OPTUNA_N_TRIALS_DL} trials, n_jobs={OPTUNA_N_JOBS})')
vae_study = run_optuna_study(_vae_km_objective, OPTUNA_N_TRIALS_DL, OPTUNA_N_JOBS)

best_vae_param = vae_study.best_trial.params
km_vae_best = MiniBatchKMeans(
    n_clusters=best_vae_param["n_clusters"],
    batch_size=10_000,
    n_init=best_vae_param["n_init"],
    random_state=RANDOM_STATE,
)
raw = km_vae_best.fit_predict(X_vae_latent)
best_vae_labels = relabel_sequential(merge_small_clusters(raw, X_vae_latent, MIN_CLUSTER_COUNT))
best_vae_stats  = compute_cluster_stats(best_vae_labels, y)
best_vae_cost   = cost_function(best_vae_labels, y)

print(f'  최적 파라미터: n_clusters={best_vae_param["n_clusters"]}, n_init={best_vae_param["n_init"]}')
print_cluster_stats(best_vae_stats, 'VAE+KMeans')
plot_alignment_result(best_vae_labels, y, 'VAE + KMeans')

## 10. 방법 비교 & 최종 선택

In [ ]:
dt_cost   = cost_function(best_dt_labels,   y)
km_cost   = cost_function(best_km_labels,   y)
ae_cost   = cost_function(best_ae_labels,   y)
gmm_cost  = cost_function(best_gmm_labels,  y)
bkm_cost  = cost_function(best_bkm_labels,  y)
agg_cost  = cost_function(best_agg_labels,  y)
hdb_cost  = cost_function(best_hdb_labels,  y)
spec_cost = cost_function(best_spec_labels, y)
iso_cost  = cost_function(best_iso_labels,  y)
vae_cost  = cost_function(best_vae_labels,  y)

comparison = pd.DataFrame([
    {'Method': 'Decision Tree',         'Param': f'max_leaves={best_dt_param}',
     'n_clusters': best_dt_stats['n_clusters'],   'min_count': best_dt_stats['min_count'],
     'aligned_4σ% (↓)': round(dt_cost,  4) if dt_cost  < float('inf') else 'INFEASIBLE',
     'mean_4σ% [ref]': round(best_dt_stats['weighted_mean_4spct'],  4),
     'max_4σ% [ref]':  round(best_dt_stats['max_4spct'],  4)},
    {'Method': 'K-Means',               'Param': f'k={best_km_param}',
     'n_clusters': best_km_stats['n_clusters'],   'min_count': best_km_stats['min_count'],
     'aligned_4σ% (↓)': round(km_cost,  4) if km_cost  < float('inf') else 'INFEASIBLE',
     'mean_4σ% [ref]': round(best_km_stats['weighted_mean_4spct'],  4),
     'max_4σ% [ref]':  round(best_km_stats['max_4spct'],  4)},
    {'Method': 'Autoencoder+KMeans',    'Param': f'latent={AE_LATENT_DIM}, k={best_ae_param}',
     'n_clusters': best_ae_stats['n_clusters'],   'min_count': best_ae_stats['min_count'],
     'aligned_4σ% (↓)': round(ae_cost,  4) if ae_cost  < float('inf') else 'INFEASIBLE',
     'mean_4σ% [ref]': round(best_ae_stats['weighted_mean_4spct'],  4),
     'max_4σ% [ref]':  round(best_ae_stats['max_4spct'],  4)},
    {'Method': 'GMM',                   'Param': f'k={best_gmm_param}',
     'n_clusters': best_gmm_stats['n_clusters'],  'min_count': best_gmm_stats['min_count'],
     'aligned_4σ% (↓)': round(gmm_cost, 4) if gmm_cost < float('inf') else 'INFEASIBLE',
     'mean_4σ% [ref]': round(best_gmm_stats['weighted_mean_4spct'], 4),
     'max_4σ% [ref]':  round(best_gmm_stats['max_4spct'], 4)},
    {'Method': 'Bisecting K-Means',     'Param': f'k={best_bkm_param}',
     'n_clusters': best_bkm_stats['n_clusters'],  'min_count': best_bkm_stats['min_count'],
     'aligned_4σ% (↓)': round(bkm_cost, 4) if bkm_cost < float('inf') else 'INFEASIBLE',
     'mean_4σ% [ref]': round(best_bkm_stats['weighted_mean_4spct'], 4),
     'max_4σ% [ref]':  round(best_bkm_stats['max_4spct'], 4)},
    {'Method': 'Agglomerative',         'Param': f'k={best_agg_param}',
     'n_clusters': best_agg_stats['n_clusters'],  'min_count': best_agg_stats['min_count'],
     'aligned_4σ% (↓)': round(agg_cost, 4) if agg_cost < float('inf') else 'INFEASIBLE',
     'mean_4σ% [ref]': round(best_agg_stats['weighted_mean_4spct'], 4),
     'max_4σ% [ref]':  round(best_agg_stats['max_4spct'], 4)},
    {'Method': 'HDBSCAN',               'Param': f'min_size={best_hdb_param}',
     'n_clusters': best_hdb_stats['n_clusters'],  'min_count': best_hdb_stats['min_count'],
     'aligned_4σ% (↓)': round(hdb_cost, 4) if hdb_cost < float('inf') else 'INFEASIBLE',
     'mean_4σ% [ref]': round(best_hdb_stats['weighted_mean_4spct'], 4),
     'max_4σ% [ref]':  round(best_hdb_stats['max_4spct'], 4)},
    {'Method': 'Spectral',              'Param': f'k={best_spec_param}',
     'n_clusters': best_spec_stats['n_clusters'], 'min_count': best_spec_stats['min_count'],
     'aligned_4σ% (↓)': round(spec_cost,4) if spec_cost< float('inf') else 'INFEASIBLE',
     'mean_4σ% [ref]': round(best_spec_stats['weighted_mean_4spct'],4),
     'max_4σ% [ref]':  round(best_spec_stats['max_4spct'],4)},
    {'Method': 'IsoForest+KMeans',      'Param': f'k={best_iso_param}',
     'n_clusters': best_iso_stats['n_clusters'],  'min_count': best_iso_stats['min_count'],
     'aligned_4σ% (↓)': round(iso_cost, 4) if iso_cost < float('inf') else 'INFEASIBLE',
     'mean_4σ% [ref]': round(best_iso_stats['weighted_mean_4spct'], 4),
     'max_4σ% [ref]':  round(best_iso_stats['max_4spct'], 4)},
    {'Method': 'VAE+KMeans',            'Param': f'latent={VAE_LATENT_DIM}, k={best_vae_param}',
     'n_clusters': best_vae_stats['n_clusters'],  'min_count': best_vae_stats['min_count'],
     'aligned_4σ% (↓)': round(vae_cost, 4) if vae_cost < float('inf') else 'INFEASIBLE',
     'mean_4σ% [ref]': round(best_vae_stats['weighted_mean_4spct'], 4),
     'max_4σ% [ref]':  round(best_vae_stats['max_4spct'], 4)},
])
print('방법 비교 (aligned 4σ% 기준):')
display(comparison)

# 최종 선택 — cost = aligned 4σ range % (낮을수록 좋음)
feasible = [
    (best_dt_labels,   best_dt_stats,   'Decision Tree',         dt_cost),
    (best_km_labels,   best_km_stats,   'K-Means',               km_cost),
    (best_ae_labels,   best_ae_stats,   'Autoencoder+KMeans',    ae_cost),
    (best_gmm_labels,  best_gmm_stats,  'GMM',                   gmm_cost),
    (best_bkm_labels,  best_bkm_stats,  'Bisecting K-Means',     bkm_cost),
    (best_agg_labels,  best_agg_stats,  'Agglomerative',         agg_cost),
    (best_hdb_labels,  best_hdb_stats,  'HDBSCAN',               hdb_cost),
    (best_spec_labels, best_spec_stats, 'Spectral',              spec_cost),
    (best_iso_labels,  best_iso_stats,  'IsoForest+KMeans',      iso_cost),
    (best_vae_labels,  best_vae_stats,  'VAE+KMeans',            vae_cost),
]
feasible = [(l, s, n, c) for l, s, n, c in feasible if c < float('inf')]
FINAL_LABELS, FINAL_STATS, FINAL_METHOD, FINAL_COST = min(feasible, key=lambda x: x[3])
print(f'\n최종 선택: {FINAL_METHOD}  (Cost={FINAL_COST:.4f}%)')


## 11. CD Median Alignment 효과 분석

> 각 클러스터의 CD median을 **전체 median CD**로 이동시켰을 때  
> (OPC recipe로 클러스터별 systematic offset을 보정했다고 가정)  
> 전체 4σ%가 얼마나 개선되는지 정량 분석

In [ ]:
def compute_median_aligned_4sigma_range(
    labels: np.ndarray,
    cd: np.ndarray,
    overall_median: float = OVERALL_MEDIAN_CD,
) -> dict:
    """클러스터별 CD median → overall_median으로 shift 후 전체 4σ range % 계산.

    Shift: CD_aligned_i = CD_i - cluster_median_i + overall_median
    → 클러스터 내 σ는 불변, 클러스터 간 위치(median) 차이만 제거
    → OPC recipe로 systematic offset 보정 효과 시뮬레이션
    """
    aligned_cd = cd.copy()
    offsets = {}
    for lbl in np.unique(labels):
        mask   = labels == lbl
        cl_med = float(np.median(cd[mask]))
        shift  = overall_median - cl_med
        aligned_cd[mask] += shift
        offsets[int(lbl)] = round(shift, 4)

    before_4spct  = compute_4sigma_range_pct(cd,         ref_median=overall_median)
    aligned_4spct = compute_4sigma_range_pct(aligned_cd, ref_median=overall_median)
    imp_abs       = before_4spct - aligned_4spct
    imp_rel       = imp_abs / before_4spct * 100

    return {
        'aligned_cd'       : aligned_cd,
        'before_4spct'     : before_4spct,
        'aligned_4spct'    : aligned_4spct,
        'improvement_abs'  : imp_abs,
        'improvement_rel'  : imp_rel,
        'cluster_offsets'  : offsets,
        'n_clusters'       : len(offsets),
    }


align_dt    = compute_median_aligned_4sigma_range(best_dt_labels,  y)
align_km    = compute_median_aligned_4sigma_range(best_km_labels,  y)
align_ae    = compute_median_aligned_4sigma_range(best_ae_labels,  y)
align_gmm   = compute_median_aligned_4sigma_range(best_gmm_labels, y)
align_bkm   = compute_median_aligned_4sigma_range(best_bkm_labels, y)
align_agg   = compute_median_aligned_4sigma_range(best_agg_labels, y)
align_hdb   = compute_median_aligned_4sigma_range(best_hdb_labels, y)
align_spec  = compute_median_aligned_4sigma_range(best_spec_labels, y)
align_iso   = compute_median_aligned_4sigma_range(best_iso_labels, y)
align_vae   = compute_median_aligned_4sigma_range(best_vae_labels, y)
align_final = compute_median_aligned_4sigma_range(FINAL_LABELS,   y)

_all_align = [
    ('Decision Tree',    align_dt),
    ('K-Means',          align_km),
    ('AE+KMeans (DL)',   align_ae),
    ('GMM',              align_gmm),
    ('Bisecting KMeans', align_bkm),
    ('Agglomerative',    align_agg),
    ('HDBSCAN',          align_hdb),
    ('Spectral',         align_spec),
    ('IsoForest+KMeans', align_iso),
    ('VAE+KMeans',       align_vae),
]

print('=' * 65)
print('  CD Median Alignment 효과 분석')
print('  (클러스터별 median → overall median으로 shift)')
print('=' * 65)
print(f'  전체 CD median                    : {OVERALL_MEDIAN_CD:.4f} nm')
print(f'  Alignment 전 4σ range %           : {BASELINE_4SIGMA_RANGE_PCT:.4f} %')
print()
for name, res in _all_align:
    print(f'  [{name}]')
    print(f'    Alignment 후 4σ range %       : {res["aligned_4spct"]:.4f} %')
    print(f'    개선량 (abs)                  : {res["improvement_abs"]:.4f} % point')
    print(f'    상대 개선율                   : {res["improvement_rel"]:.2f} %')
    print()

In [ ]:
# ── 11-1. Alignment 전후 4σ% 비교 시각화 (10종) ──────────────────
methods_align = [name for name, _ in _all_align]
results_align = [res  for _, res  in _all_align]

before_vals  = [BASELINE_4SIGMA_RANGE_PCT] * len(methods_align)
after_vals   = [r['aligned_4spct']   for r in results_align]
imp_abs_vals = [r['improvement_abs'] for r in results_align]

x = np.arange(len(methods_align))
w = 0.35

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

bars_before = axes[0].bar(x - w/2, before_vals, w, label='Before Alignment (baseline)', color='#aec7e8', edgecolor='k', lw=0.5)
bars_after  = axes[0].bar(x + w/2, after_vals,  w, label='After Alignment',             color='#2ca02c', edgecolor='k', lw=0.5, alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(methods_align, rotation=30, ha='right')
axes[0].set_ylabel('4σ range %')
axes[0].set_title('CD Median Alignment: Before vs After (10 Methods)', fontweight='bold')
axes[0].legend()
for i, (b, a) in enumerate(zip(before_vals, after_vals)):
    axes[0].text(x[i]-w/2, b+0.001, f'{b:.3f}', ha='center', va='bottom', fontsize=7)
    axes[0].text(x[i]+w/2, a+0.001, f'{a:.3f}', ha='center', va='bottom', fontsize=7, color='darkgreen')

bars_imp = axes[1].bar(x, imp_abs_vals, color='#ff7f0e', edgecolor='k', lw=0.5, alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(methods_align, rotation=30, ha='right')
axes[1].set_ylabel('Improvement (% point)')
axes[1].set_title('Alignment Improvement (abs) per Method', fontweight='bold')
for bar, val in zip(bars_imp, imp_abs_vals):
    axes[1].text(bar.get_x()+bar.get_width()/2, val+0.0005, f'{val:.4f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/10_alignment_comparison.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── 11-2. 방법별 산포 히스토그램: Alignment 전/후 비교 (10종) ────────
# 각 방법마다 원본 CD 분포(파란색)와 median alignment 후 예상 분포(초록색)를
# 함께 plot하여 4σ range 개선량을 시각적으로 확인

method_configs = [
    ('Decision Tree',       best_dt_labels,   align_dt),
    ('K-Means',             best_km_labels,   align_km),
    ('AE + K-Means (DL)',   best_ae_labels,   align_ae),
    ('GMM',                 best_gmm_labels,  align_gmm),
    ('Bisecting KMeans',    best_bkm_labels,  align_bkm),
    ('Agglomerative',       best_agg_labels,  align_agg),
    ('HDBSCAN',             best_hdb_labels,  align_hdb),
    ('Spectral',            best_spec_labels, align_spec),
    ('IsoForest+KMeans',    best_iso_labels,  align_iso),
    ('VAE + K-Means',       best_vae_labels,  align_vae),
]

n_methods = len(method_configs)
ncols = 5
nrows = (n_methods + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4), sharey=False)
axes_flat = axes.ravel()
fig.suptitle('CD Distribution: Before vs After Median Alignment (10 Methods)',
             fontsize=14, fontweight='bold', y=1.02)

for ax, (name, labels, res) in zip(axes_flat, method_configs):
    cd_before = y
    cd_after  = res['aligned_cd']

    x_min = min(cd_before.min(), cd_after.min())
    x_max = max(cd_before.max(), cd_after.max())
    bins  = np.linspace(x_min, x_max, 80)

    ax.hist(cd_before, bins=bins, alpha=0.55, color='steelblue', density=True, label='Before')
    ax.hist(cd_after,  bins=bins, alpha=0.55, color='seagreen',  density=True, label='After')

    b4 = res['before_4spct']
    a4 = res['aligned_4spct']
    ax.set_title(f'{name}\nBefore: {b4:.3f}%  →  After: {a4:.3f}%', fontsize=9, fontweight='bold')
    ax.set_xlabel('CD (nm)', fontsize=8)
    ax.set_ylabel('Density', fontsize=8)
    ax.legend(fontsize=7)

# Hide unused subplots
for ax in axes_flat[n_methods:]:
    ax.set_visible(False)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/10b_alignment_histograms.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── 11-2. 최종 방법의 Alignment 상세 분석 ────────────────────
print(f'최종 방법 [{FINAL_METHOD}] Alignment 상세 분석')

# Before/After CD 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 전체 CD 분포 비교
bins = np.linspace(y.min(), y.max(), 150)
axes[0].hist(y, bins=bins, alpha=0.5, color='steelblue', label=f'Before  (4σ%={BASELINE_4SIGMA_RANGE_PCT:.3f}%)', density=True)
axes[0].hist(align_final['aligned_cd'], bins=bins, alpha=0.5, color='seagreen',
             label=f'After   (4σ%={align_final["aligned_4spct"]:.3f}%)', density=True)
axes[0].axvline(OVERALL_MEDIAN_CD, color='red', ls='--', lw=1.5, label=f'Overall Median={OVERALL_MEDIAN_CD:.2f}nm')
axes[0].set_title('CD Distribution: Before vs After Alignment', fontweight='bold')
axes[0].set_xlabel('CD (nm)'); axes[0].set_ylabel('Density'); axes[0].legend()

# 클러스터별 shift 크기
offsets     = align_final['cluster_offsets']
offset_vals = list(offsets.values())
cluster_ids = list(offsets.keys())
bar_c = ['seagreen' if v > 0 else 'salmon' for v in offset_vals]
axes[1].bar([str(c) for c in cluster_ids], offset_vals, color=bar_c, edgecolor='k', lw=0.3, alpha=0.85)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('Cluster Median Shift Amount (nm)\n(green: +shift, red: -shift)', fontweight='bold')
axes[1].set_xlabel('Cluster'); axes[1].set_ylabel('Shift (nm)')
if len(cluster_ids) > 20:
    axes[1].tick_params(axis='x', labelsize=7, rotation=45)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/09_alignment_detail.png', bbox_inches='tight')
plt.show()

print(f'\n  Alignment 전 4σ%  : {align_final["before_4spct"]:.4f} %')
print(f'  Alignment 후 4σ%  : {align_final["aligned_4spct"]:.4f} %')
print(f'  개선량            : {align_final["improvement_abs"]:.4f} % point')
print(f'  상대 개선율       : {align_final["improvement_rel"]:.2f} %')

In [ ]:
# ── 11-3. 클러스터별 Before/After σ & 4σ% 비교 ───────────────
n_cls = FINAL_STATS['n_clusters']

cluster_ids_sorted = sorted(FINAL_STATS['cluster_counts'].keys())
before_sigma_per  = [FINAL_STATS['sigma_per_cluster'][c] for c in cluster_ids_sorted]  # nm σ
before_4spct_per  = [FINAL_STATS['four_sigma_range_pct'][c]    for c in cluster_ids_sorted]
cluster_counts_   = [FINAL_STATS['cluster_counts'][c]    for c in cluster_ids_sorted]
cluster_medians_  = [FINAL_STATS['median_per_cluster'][c] for c in cluster_ids_sorted]

# After alignment: 각 클러스터 내부 분산은 동일 (shift만 했으므로)
# → 클러스터 내 σ는 변하지 않음. 전체 4σ%만 변화.
# 여기서는 클러스터별 CD median offset을 시각화하여 보정 필요량 표시

fig, axes = plt.subplots(2, 1, figsize=(max(12, n_cls * 0.7), 10))

x_pos = np.arange(len(cluster_ids_sorted))
bar_c = cm.RdYlGn_r(np.array(before_4spct_per) / max(before_4spct_per))
axes[0].bar(x_pos, before_4spct_per, color=bar_c, edgecolor='white', lw=0.4)
axes[0].axhline(FINAL_STATS['weighted_mean_4spct'], color='blue',  ls='--', lw=2,
                label=f'Weighted Mean 4σ% = {FINAL_STATS["weighted_mean_4spct"]:.3f}%')
axes[0].axhline(BASELINE_4SIGMA_RANGE_PCT, color='gray', ls=':',  lw=2,
                label=f'Baseline 4σ% = {BASELINE_4SIGMA_RANGE_PCT:.3f}%')
axes[0].axhline(FINAL_STATS['max_4spct'], color='red', ls='-.', lw=2,
                label=f'Max 4σ% = {FINAL_STATS["max_4spct"]:.3f}%')
axes[0].set_title(f'Per-Cluster 4σ% [{FINAL_METHOD}]', fontweight='bold')
axes[0].set_ylabel('4σ% = 4σ/median_CD × 100 (%)'); axes[0].legend()
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([f'C{c}\n(n={cluster_counts_[i]:,})' for i, c in enumerate(cluster_ids_sorted)],
                         fontsize=8, rotation=45, ha='right')

# Cluster median vs overall median (offset 크기 = alignment에서 보정해야 할 양)
offsets_nm = [m - OVERALL_MEDIAN_CD for m in cluster_medians_]
bar_c2 = ['seagreen' if v >= 0 else 'salmon' for v in offsets_nm]
axes[1].bar(x_pos, offsets_nm, color=bar_c2, edgecolor='white', lw=0.4, alpha=0.85)
axes[1].axhline(0, color='black', lw=1)
axes[1].set_title('Cluster CD Median Offset from Overall Median (= Alignment Shift)', fontweight='bold')
axes[1].set_ylabel('Offset (nm) = Cluster Median − Overall Median')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f'C{c}' for c in cluster_ids_sorted], fontsize=8, rotation=45, ha='right')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/10_per_cluster_4spct.png', bbox_inches='tight')
plt.show()

## 12. 전체 방법 통합 비교

In [ ]:
# 10가지 방법 + baseline 종합 비교
all_methods = {
    'Baseline'           : {'4spct_before': BASELINE_4SIGMA_RANGE_PCT, '4spct_after': BASELINE_4SIGMA_RANGE_PCT,
                             'mean_4spct': BASELINE_4SIGMA_RANGE_PCT, 'max_4spct': BASELINE_4SIGMA_RANGE_PCT,
                             'n_clusters': 1, 'cost': BASELINE_4SIGMA_RANGE_PCT},
    'Decision Tree'      : {'4spct_before': BASELINE_4SIGMA_RANGE_PCT, '4spct_after': align_dt['aligned_4spct'],
                             'mean_4spct': best_dt_stats['weighted_mean_4spct'],
                             'max_4spct': best_dt_stats['max_4spct'],
                             'n_clusters': best_dt_stats['n_clusters'], 'cost': dt_cost},
    'K-Means'            : {'4spct_before': BASELINE_4SIGMA_RANGE_PCT, '4spct_after': align_km['aligned_4spct'],
                             'mean_4spct': best_km_stats['weighted_mean_4spct'],
                             'max_4spct': best_km_stats['max_4spct'],
                             'n_clusters': best_km_stats['n_clusters'], 'cost': km_cost},
    'AE+KMeans (DL)'     : {'4spct_before': BASELINE_4SIGMA_RANGE_PCT, '4spct_after': align_ae['aligned_4spct'],
                             'mean_4spct': best_ae_stats['weighted_mean_4spct'],
                             'max_4spct': best_ae_stats['max_4spct'],
                             'n_clusters': best_ae_stats['n_clusters'], 'cost': ae_cost},
    'GMM'                : {'4spct_before': BASELINE_4SIGMA_RANGE_PCT, '4spct_after': align_gmm['aligned_4spct'],
                             'mean_4spct': best_gmm_stats['weighted_mean_4spct'],
                             'max_4spct': best_gmm_stats['max_4spct'],
                             'n_clusters': best_gmm_stats['n_clusters'], 'cost': gmm_cost},
    'Bisecting KMeans'   : {'4spct_before': BASELINE_4SIGMA_RANGE_PCT, '4spct_after': align_bkm['aligned_4spct'],
                             'mean_4spct': best_bkm_stats['weighted_mean_4spct'],
                             'max_4spct': best_bkm_stats['max_4spct'],
                             'n_clusters': best_bkm_stats['n_clusters'], 'cost': bkm_cost},
    'Agglomerative'      : {'4spct_before': BASELINE_4SIGMA_RANGE_PCT, '4spct_after': align_agg['aligned_4spct'],
                             'mean_4spct': best_agg_stats['weighted_mean_4spct'],
                             'max_4spct': best_agg_stats['max_4spct'],
                             'n_clusters': best_agg_stats['n_clusters'], 'cost': agg_cost},
    'HDBSCAN'            : {'4spct_before': BASELINE_4SIGMA_RANGE_PCT, '4spct_after': align_hdb['aligned_4spct'],
                             'mean_4spct': best_hdb_stats['weighted_mean_4spct'],
                             'max_4spct': best_hdb_stats['max_4spct'],
                             'n_clusters': best_hdb_stats['n_clusters'], 'cost': hdb_cost},
    'Spectral'           : {'4spct_before': BASELINE_4SIGMA_RANGE_PCT, '4spct_after': align_spec['aligned_4spct'],
                             'mean_4spct': best_spec_stats['weighted_mean_4spct'],
                             'max_4spct': best_spec_stats['max_4spct'],
                             'n_clusters': best_spec_stats['n_clusters'], 'cost': spec_cost},
    'IsoForest+KMeans'   : {'4spct_before': BASELINE_4SIGMA_RANGE_PCT, '4spct_after': align_iso['aligned_4spct'],
                             'mean_4spct': best_iso_stats['weighted_mean_4spct'],
                             'max_4spct': best_iso_stats['max_4spct'],
                             'n_clusters': best_iso_stats['n_clusters'], 'cost': iso_cost},
    'VAE+KMeans'         : {'4spct_before': BASELINE_4SIGMA_RANGE_PCT, '4spct_after': align_vae['aligned_4spct'],
                             'mean_4spct': best_vae_stats['weighted_mean_4spct'],
                             'max_4spct': best_vae_stats['max_4spct'],
                             'n_clusters': best_vae_stats['n_clusters'], 'cost': vae_cost},
}

import matplotlib.cm as cm_mod
n_methods_plot = len(all_methods)
colors_m = cm_mod.tab20(np.linspace(0, 1, n_methods_plot))
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
names = list(all_methods.keys())
best_marker = FINAL_METHOD

mean_vals = [v['mean_4spct'] for v in all_methods.values()]
bars0 = axes[0].bar(names, mean_vals, color=colors_m, edgecolor='k', lw=0.5, alpha=0.85)
axes[0].set_title('Cluster Mean 4σ% (within cluster)', fontweight='bold')
axes[0].set_ylabel('4σ% (%)')
axes[0].tick_params(axis='x', rotation=40)
for bar, val in zip(bars0, mean_vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, val+0.002, f'{val:.3f}%', ha='center', va='bottom', fontsize=7, rotation=60)

after_vals2 = [v['4spct_after'] for v in all_methods.values()]
bars1 = axes[1].bar(names, after_vals2, color=colors_m, edgecolor='k', lw=0.5, alpha=0.85)
axes[1].set_title('Overall 4σ% After Median Alignment', fontweight='bold')
axes[1].set_ylabel('4σ% (%)')
axes[1].tick_params(axis='x', rotation=40)
for bar, val in zip(bars1, after_vals2):
    axes[1].text(bar.get_x()+bar.get_width()/2, val+0.002, f'{val:.3f}%', ha='center', va='bottom', fontsize=7, rotation=60)

costs_v = [v['cost'] if v['cost'] < float('inf') else 0 for v in all_methods.values()]
bars2 = axes[2].bar(names, costs_v, color=colors_m, edgecolor='k', lw=0.5, alpha=0.85)
axes[2].set_title('Cost = aligned 4σ range % after median alignment', fontweight='bold')
axes[2].set_ylabel('Cost')
axes[2].tick_params(axis='x', rotation=40)
for bar, val, nm in zip(bars2, costs_v, names):
    label = f'{val:.4f}' + (' ★' if nm == best_marker else '')
    axes[2].text(bar.get_x()+bar.get_width()/2, val+0.001, label, ha='center', va='bottom', fontsize=7,
                  fontweight='bold' if nm == best_marker else 'normal', rotation=60)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/11_full_comparison.png', bbox_inches='tight')
plt.show()


## 13. 클러스터 시각화 (최종 방법)

In [ ]:
# ── 13-1. 클러스터별 CD 분포 (Box + σ bar) ───────────────────
n_cls    = FINAL_STATS['n_clusters']
cl_ids   = sorted(FINAL_STATS['cluster_counts'].keys())
cl_data  = [y[FINAL_LABELS == c] for c in cl_ids]
cl_4spct = [compute_4sigma_range_pct(cl_data[i], ref_median=OVERALL_MEDIAN_CD) for i in range(n_cls)]
cl_size  = [len(d) for d in cl_data]
order    = np.argsort([d.mean() for d in cl_data])

fig, axes = plt.subplots(2, 1, figsize=(max(12, n_cls * 0.7), 11))

bp = axes[0].boxplot([cl_data[order[i]] for i in range(n_cls)],
                      notch=True, patch_artist=True, showfliers=False)
for patch, c in zip(bp['boxes'], cm.RdYlGn_r(np.linspace(0.1, 0.9, n_cls))):
    patch.set_facecolor(c); patch.set_alpha(0.7)
axes[0].set_xticks(range(1, n_cls+1))
axes[0].set_xticklabels([f'C{cl_ids[order[i]]}\n(n={cl_size[order[i]]:,})' for i in range(n_cls)],
                          fontsize=8, rotation=45, ha='right')
axes[0].axhline(y.mean(), color='navy', ls='--', lw=1.2, label='Overall Mean')
axes[0].axhline(OVERALL_MEDIAN_CD, color='red', ls=':', lw=1.2, label='Overall Median')
axes[0].set_title(f'{FINAL_METHOD}: CD Distribution per Cluster', fontweight='bold')
axes[0].set_ylabel('CD (nm)'); axes[0].legend()

spct_ord = [cl_4spct[order[i]] for i in range(n_cls)]
bar_c = cm.RdYlGn_r(np.array(spct_ord)/max(spct_ord))
axes[1].bar(range(n_cls), spct_ord, color=bar_c, edgecolor='white', lw=0.4)
axes[1].axhline(FINAL_STATS['weighted_mean_4spct'], color='blue', ls='--', lw=2,
                label=f'Mean 4σ% = {FINAL_STATS["weighted_mean_4spct"]:.3f}%')
axes[1].axhline(BASELINE_4SIGMA_RANGE_PCT, color='gray', ls=':', lw=2,
                label=f'Baseline 4σ% = {BASELINE_4SIGMA_RANGE_PCT:.3f}%')
axes[1].axhline(FINAL_STATS['max_4spct'], color='red', ls='-.', lw=2,
                label=f'Max 4σ% = {FINAL_STATS["max_4spct"]:.3f}%')
axes[1].set_xticks(range(n_cls))
axes[1].set_xticklabels([f'C{cl_ids[order[i]]}' for i in range(n_cls)], fontsize=8, rotation=45, ha='right')
axes[1].set_title('Per-Cluster 4σ% = 4σ/median_CD × 100', fontweight='bold')
axes[1].set_ylabel('4σ% (%)'); axes[1].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/12_cluster_cd_4spct.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 13-2. PCA 2D 시각화 ────────────────────────────────────────
pca_n = min(20_000, len(X_sel))
pca_i = np.random.RandomState(RANDOM_STATE).choice(len(X_sel), pca_n, replace=False)
pca   = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_sel[pca_i])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
palette = cm.tab20(np.linspace(0, 1, n_cls))
for c in range(n_cls):
    m = FINAL_LABELS[pca_i] == c
    axes[0].scatter(X_pca[m, 0], X_pca[m, 1], s=3, alpha=0.4, color=palette[c], label=f'C{c}')
axes[0].set_title(f'PCA 2D — {FINAL_METHOD}', fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
if n_cls <= 20: axes[0].legend(markerscale=3, fontsize=8, ncol=2)

sc = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y[pca_i], cmap='plasma', s=3, alpha=0.5,
                      vmin=np.percentile(y, 2), vmax=np.percentile(y, 98))
plt.colorbar(sc, ax=axes[1], label='CD (nm)')
axes[1].set_title('PCA 2D — CD Value', fontweight='bold')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/13_pca_visualization.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 13-3. Latent 공간 PCA (AE용) ──────────────────────────────
pca_lat = PCA(n_components=2, random_state=RANDOM_STATE)
X_lat_pca = pca_lat.fit_transform(X_latent[pca_i])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
ae_n_cls = best_ae_stats['n_clusters']
palette2 = cm.tab20(np.linspace(0, 1, ae_n_cls))
for c in range(ae_n_cls):
    m = best_ae_labels[pca_i] == c
    if m.sum() > 0:
        axes[0].scatter(X_lat_pca[m, 0], X_lat_pca[m, 1], s=3, alpha=0.4, color=palette2[c], label=f'C{c}')
axes[0].set_title(f'Latent Space PCA — AE+KMeans (k={best_ae_param})', fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca_lat.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca_lat.explained_variance_ratio_[1]*100:.1f}%)')
if ae_n_cls <= 20: axes[0].legend(markerscale=3, fontsize=8, ncol=2)

sc2 = axes[1].scatter(X_lat_pca[:, 0], X_lat_pca[:, 1], c=y[pca_i], cmap='plasma', s=3, alpha=0.5,
                       vmin=np.percentile(y, 2), vmax=np.percentile(y, 98))
plt.colorbar(sc2, ax=axes[1], label='CD (nm)')
axes[1].set_title('Latent Space PCA — CD Value', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/14_latent_pca.png', bbox_inches='tight')
plt.show()

## 14. 결과 저장

In [ ]:
# 최종 레이블 + 클러스터별 통계를 원본 df에 붙이기
df_out = df_clean.copy()
df_out['cluster']          = FINAL_LABELS
df_out['cluster_sigma_nm'] = df_out['cluster'].map(FINAL_STATS['sigma_per_cluster'])  # nm σ
df_out['cluster_4spct']    = df_out['cluster'].map(FINAL_STATS['four_sigma_range_pct'])
df_out['cluster_median_cd']= df_out['cluster'].map(FINAL_STATS['median_per_cluster'])
df_out['align_shift_nm']   = OVERALL_MEDIAN_CD - df_out['cluster_median_cd']

df_out.to_parquet(f'{OUTPUT_DIR}/clustered_data.parquet', index=True)
shap_importance.to_csv(f'{OUTPUT_DIR}/shap_importance.csv', header=['mean_abs_shap'])

summary = {
    'final_method'                   : FINAL_METHOD,
    'n_clusters'                     : int(FINAL_STATS['n_clusters']),
    'min_cluster_count'              : int(FINAL_STATS['min_count']),
    'selected_features'              : selected_features,
    'overall_median_cd_nm'           : OVERALL_MEDIAN_CD,
    'cost_alpha'                     : ALPHA,
    'cost_beta'                      : BETA,
    'cost_value'                     : round(FINAL_COST, 6),
    'baseline_4sigma_pct'            : round(BASELINE_4SIGMA_RANGE_PCT, 6),
    'cluster_weighted_mean_4spct'    : round(FINAL_STATS['weighted_mean_4spct'], 6),
    'cluster_max_4spct'              : round(FINAL_STATS['max_4spct'], 6),
    'alignment_before_4spct'         : round(align_final['before_4spct'], 6),
    'alignment_after_4spct'          : round(align_final['aligned_4spct'], 6),
    'alignment_improvement_abs_ppt'  : round(align_final['improvement_abs'], 6),
    'alignment_improvement_rel_pct'  : round(align_final['improvement_rel'], 4),
    'method_comparison': {
        'Decision Tree'   : {'cost': round(dt_cost,  4) if dt_cost  < float('inf') else None,
                              'mean_4spct': round(best_dt_stats['weighted_mean_4spct'],  4),
                              'max_4spct' : round(best_dt_stats['max_4spct'],  4),
                              'align_after_4spct': round(align_dt['aligned_4spct'],  4)},
        'K-Means'         : {'cost': round(km_cost,  4) if km_cost  < float('inf') else None,
                              'mean_4spct': round(best_km_stats['weighted_mean_4spct'],  4),
                              'max_4spct' : round(best_km_stats['max_4spct'],  4),
                              'align_after_4spct': round(align_km['aligned_4spct'],  4)},
        'AE+KMeans (DL)'  : {'cost': round(ae_cost,  4) if ae_cost  < float('inf') else None,
                              'mean_4spct': round(best_ae_stats['weighted_mean_4spct'],  4),
                              'max_4spct' : round(best_ae_stats['max_4spct'],  4),
                              'align_after_4spct': round(align_ae['aligned_4spct'],  4)},
        'GMM'             : {'cost': round(gmm_cost, 4) if gmm_cost < float('inf') else None,
                              'mean_4spct': round(best_gmm_stats['weighted_mean_4spct'], 4),
                              'max_4spct' : round(best_gmm_stats['max_4spct'], 4),
                              'align_after_4spct': round(align_gmm['aligned_4spct'], 4)},
        'Bisecting KMeans': {'cost': round(bkm_cost, 4) if bkm_cost < float('inf') else None,
                              'mean_4spct': round(best_bkm_stats['weighted_mean_4spct'], 4),
                              'max_4spct' : round(best_bkm_stats['max_4spct'], 4),
                              'align_after_4spct': round(align_bkm['aligned_4spct'], 4)},
        'Agglomerative'   : {'cost': round(agg_cost, 4) if agg_cost < float('inf') else None,
                              'mean_4spct': round(best_agg_stats['weighted_mean_4spct'], 4),
                              'max_4spct' : round(best_agg_stats['max_4spct'], 4),
                              'align_after_4spct': round(align_agg['aligned_4spct'], 4)},
        'HDBSCAN'         : {'cost': round(hdb_cost, 4) if hdb_cost < float('inf') else None,
                              'mean_4spct': round(best_hdb_stats['weighted_mean_4spct'], 4),
                              'max_4spct' : round(best_hdb_stats['max_4spct'], 4),
                              'align_after_4spct': round(align_hdb['aligned_4spct'], 4)},
        'Spectral'        : {'cost': round(spec_cost,4) if spec_cost< float('inf') else None,
                              'mean_4spct': round(best_spec_stats['weighted_mean_4spct'],4),
                              'max_4spct' : round(best_spec_stats['max_4spct'], 4),
                              'align_after_4spct': round(align_spec['aligned_4spct'], 4)},
        'IsoForest+KMeans': {'cost': round(iso_cost, 4) if iso_cost < float('inf') else None,
                              'mean_4spct': round(best_iso_stats['weighted_mean_4spct'], 4),
                              'max_4spct' : round(best_iso_stats['max_4spct'], 4),
                              'align_after_4spct': round(align_iso['aligned_4spct'], 4)},
        'VAE+KMeans'      : {'cost': round(vae_cost, 4) if vae_cost < float('inf') else None,
                              'mean_4spct': round(best_vae_stats['weighted_mean_4spct'], 4),
                              'max_4spct' : round(best_vae_stats['max_4spct'], 4),
                              'align_after_4spct': round(align_vae['aligned_4spct'], 4)},
    },
    'config': {
        'MIN_CLUSTER_COUNT': MIN_CLUSTER_COUNT,
        'SHAP_TOP_K'       : SHAP_TOP_K,
        'AE_LATENT_DIM'    : AE_LATENT_DIM,
        'AE_EPOCHS'        : AE_EPOCHS,
        'RANDOM_STATE'     : RANDOM_STATE,
    },
}
with open(f'{OUTPUT_DIR}/clustering_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print('결과 저장 완료:')
for f in sorted(Path(OUTPUT_DIR).iterdir()):
    print(f'  {f.name}')

In [ ]:
# ── 최종 요약 출력 ─────────────────────────────────────────────
print('=' * 68)
print('  In-House Layout Clustering — 최종 결과 요약')
print('=' * 68)
print(f'  CD median (전체)          : {OVERALL_MEDIAN_CD:.4f} nm')
print(f'  기준선 4σ%                 : {BASELINE_4SIGMA_RANGE_PCT:.4f} %')
print()
print(f'  [최적 방법: {FINAL_METHOD}]')
print(f'  클러스터 수                : {FINAL_STATS["n_clusters"]}')
print(f'  최소 클러스터 크기         : {FINAL_STATS["min_count"]:,} ≥ {MIN_CLUSTER_COUNT}')
print(f'  Cost (aligned 4σ%)         : {FINAL_COST:.4f} %')
print(f'  개선율                     : {(BASELINE_4SIGMA_RANGE_PCT - FINAL_COST) / BASELINE_4SIGMA_RANGE_PCT * 100:.2f} %')
print()
print(f'  [참고 — 클러스터별 분포]')
print(f'  클러스터 가중 평균 4σ%     : {FINAL_STATS["weighted_mean_4spct"]:.4f} %')
print(f'  클러스터 최대 4σ%          : {FINAL_STATS["max_4spct"]:.4f} %')
print('=' * 68)
